In [1]:
"""
UC4 Structured Micro-Priority Engine
ACCESS-DP / Adaptive Care Focus Checklist

Purpose:
- Generate personalized caregiver micro-priorities using structured patient context,
  care-plan priorities, medication watch areas, UC1/UC2/caregiver events,
  wearable summaries, and previous UC4 priorities.
- Every output priority is selected from a finite template registry.
- Every "what to log next" item has a structured input schema.
- Caregiver responses are stored as structured events and feed the next cycle.

Safety boundaries:
- UC4 does not diagnose.
- UC4 does not detect seizures.
- UC4 does not detect wounds.
- UC4 does not measure tone/spasticity.
- UC4 does not infer medication causality.
- UC4 does not recommend medication, dose, or treatment changes.
- SLM/LLM is not required and is not used for scoring.
- Free text is not used for scoring in this prototype.
"""

'\nUC4 Structured Micro-Priority Engine\nACCESS-DP / Adaptive Care Focus Checklist\n\nPurpose:\n- Generate personalized caregiver micro-priorities using structured patient context,\n  care-plan priorities, medication watch areas, UC1/UC2/caregiver events,\n  wearable summaries, and previous UC4 priorities.\n- Every output priority is selected from a finite template registry.\n- Every "what to log next" item has a structured input schema.\n- Caregiver responses are stored as structured events and feed the next cycle.\n\nSafety boundaries:\n- UC4 does not diagnose.\n- UC4 does not detect seizures.\n- UC4 does not detect wounds.\n- UC4 does not measure tone/spasticity.\n- UC4 does not infer medication causality.\n- UC4 does not recommend medication, dose, or treatment changes.\n- SLM/LLM is not required and is not used for scoring.\n- Free text is not used for scoring in this prototype.\n'

In [3]:
import json
import math
import random
from pathlib import Path
from datetime import datetime, timedelta, timezone
from collections import Counter, defaultdict

import pandas as pd
import numpy as np

OUTPUT_DIR = Path("uc4_structured_micropriority_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

SCHEMA_VERSION = "uc4_schema_v0.1.0"
TEMPLATE_REGISTRY_VERSION = "uc4_template_registry_v0.1.0"
ENGINE_VERSION = "uc4_structured_micropriority_engine_v0.1.0"

NOW = datetime(2026, 7, 16, 12, 0, tzinfo=timezone.utc)

def iso(dt):
    return dt.isoformat().replace("+00:00", "Z")

In [5]:
OBSERVATION_CODES = [
    "LOOKS_NORMAL",
    "NOT_SURE",
    "DEVICE_OR_SENSOR_ISSUE",

    "RECENT_ACTIVITY_OR_EXERTION",
    "LOW_MOVEMENT",
    "TRANSFER_OR_POSITIONING_CONTEXT",

    "PAIN_OR_DISCOMFORT",
    "UNUSUAL_FATIGUE",
    "POOR_SLEEP_OR_RESTLESSNESS",

    "BREATHING_CONCERN",
    "COLOR_OR_OXYGEN_CONCERN",

    "MISSED_OR_DELAYED_MEDICATION",
    "RECENT_MEDICATION_CHANGE",
    "APPETITE_OR_HYDRATION_CHANGE",

    "BOWEL_OR_BLADDER_CHANGE",
    "SKIN_OR_PRESSURE_CONCERN",

    "SEIZURE_LIKE_EVENT_REPORTED",
    "UNUSUAL_RESPONSIVENESS",

    "FALL_OR_NEAR_FALL",
    "THERAPY_ROUTINE_DIFFICULTY",
    "CAREGIVER_WANTS_PROVIDER_REVIEW"
]

CONTEXT_CODES = [
    "DURING_TRANSFER",
    "WHILE_SITTING_OR_POSITIONED",
    "AFTER_ACTIVITY_OR_THERAPY",
    "AROUND_MEDICATION_TIME",
    "DURING_SLEEP_OR_NIGHT",
    "MEAL_OR_HYDRATION_RELATED",
    "BATHROOM_OR_BOWEL_BLADDER",
    "RESPIRATORY_ROUTINE_RELATED",
    "THERAPY_OR_EXERCISE_RELATED",
    "UNKNOWN_OR_NOT_SURE"
]

MEDICATION_WATCH_AREA_CODES = [
    "SLEEPINESS_FATIGUE",
    "DIZZINESS_OR_LIGHTHEADEDNESS",
    "WEAKNESS_OR_LOW_TONE_CONCERN",
    "MOOD_BEHAVIOR_CHANGE",
    "APPETITE_OR_HYDRATION_CHANGE",
    "BOWEL_CHANGE",
    "BREATHING_CONCERN",
    "HEART_RATE_OR_BP_CONCERN",
    "SKIN_RASH_OR_ALLERGY_CONCERN",
    "MISSED_OR_DELAYED_DOSE",
    "MEDICATION_TIMING_CONTEXT_NEEDED"
]

CARE_PLAN_PRIORITY_CODES = [
    "MOBILITY_SUPPORT",
    "POSITIONING_SUPPORT",
    "SKIN_PRESSURE_PREVENTION",
    "MEDICATION_ADHERENCE",
    "BOWEL_ROUTINE_SUPPORT",
    "HYDRATION_SUPPORT",
    "RESPIRATORY_MONITORING",
    "SLEEP_ROUTINE_SUPPORT",
    "PROVIDER_FOLLOWUP",
    "THERAPY_REHAB_ROUTINE",
    "SEIZURE_ACTION_PLAN_SUPPORT",
    "COMMUNICATION_SUPPORT",
    "FALL_PREVENTION"
]

INPUT_TYPES = [
    "single_select",
    "multi_select",
    "boolean",
    "number",
    "time",
    "short_text",
    "medication_select"
]

vocab_manifest = {
    "schema_version": SCHEMA_VERSION,
    "observation_codes": OBSERVATION_CODES,
    "context_codes": CONTEXT_CODES,
    "medication_watch_area_codes": MEDICATION_WATCH_AREA_CODES,
    "care_plan_priority_codes": CARE_PLAN_PRIORITY_CODES,
    "input_types": INPUT_TYPES
}

with open(OUTPUT_DIR / "uc4_controlled_vocabularies.json", "w") as f:
    json.dump(vocab_manifest, f, indent=2)

vocab_manifest

{'schema_version': 'uc4_schema_v0.1.0',
 'observation_codes': ['LOOKS_NORMAL',
  'NOT_SURE',
  'DEVICE_OR_SENSOR_ISSUE',
  'RECENT_ACTIVITY_OR_EXERTION',
  'LOW_MOVEMENT',
  'TRANSFER_OR_POSITIONING_CONTEXT',
  'PAIN_OR_DISCOMFORT',
  'UNUSUAL_FATIGUE',
  'POOR_SLEEP_OR_RESTLESSNESS',
  'BREATHING_CONCERN',
  'COLOR_OR_OXYGEN_CONCERN',
  'MISSED_OR_DELAYED_MEDICATION',
  'RECENT_MEDICATION_CHANGE',
  'APPETITE_OR_HYDRATION_CHANGE',
  'BOWEL_OR_BLADDER_CHANGE',
  'SKIN_OR_PRESSURE_CONCERN',
  'SEIZURE_LIKE_EVENT_REPORTED',
  'UNUSUAL_RESPONSIVENESS',
  'FALL_OR_NEAR_FALL',
  'THERAPY_ROUTINE_DIFFICULTY',
  'CAREGIVER_WANTS_PROVIDER_REVIEW'],
 'context_codes': ['DURING_TRANSFER',
  'WHILE_SITTING_OR_POSITIONED',
  'AFTER_ACTIVITY_OR_THERAPY',
  'AROUND_MEDICATION_TIME',
  'DURING_SLEEP_OR_NIGHT',
  'MEAL_OR_HYDRATION_RELATED',
  'BATHROOM_OR_BOWEL_BLADDER',
  'RESPIRATORY_ROUTINE_RELATED',
  'THERAPY_OR_EXERCISE_RELATED',
  'UNKNOWN_OR_NOT_SURE'],
 'medication_watch_area_codes': ['SLEEPI

In [7]:
OPTION_SETS = {
    "YES_NO_NOT_SURE": ["YES", "NO", "NOT_SURE"],
    "YES_NO_PARTLY_NOT_SURE": ["YES", "NO", "PARTLY", "NOT_SURE"],
    "TIME_OF_DAY": ["MORNING", "AFTERNOON", "EVENING", "NIGHT", "NOT_SURE"],
    "BASELINE_DIFFERENCE": ["YES_DIFFERENT", "NO_USUAL", "NOT_SURE"],
    "MEDICATION_TIMING_RELATION": [
        "BEFORE_MEDICATION",
        "AFTER_MEDICATION",
        "AROUND_MEDICATION_TIME",
        "NOT_NEAR_MEDICATION_TIME",
        "NOT_SURE"
    ],
    "DOSE_STATUS": ["MISSED", "DELAYED", "TAKEN_LATER", "NOT_SURE"],
    "POSITION": [
        "SEATED_WHEELCHAIR",
        "SEATED_CHAIR",
        "LYING_BACK",
        "LYING_LEFT_SIDE",
        "LYING_RIGHT_SIDE",
        "STANDING_SUPPORTED",
        "OTHER",
        "NOT_SURE"
    ],
    "TRANSFER_PHASE": ["BEFORE_TRANSFER", "DURING_TRANSFER", "AFTER_TRANSFER", "NOT_SURE"],
    "DISCOMFORT_CUES": [
        "GRIMACING",
        "GUARDING",
        "CRYING_OR_VOCALIZING",
        "RESTLESSNESS",
        "WITHDRAWING",
        "INCREASED_TENSION",
        "CAREGIVER_NOT_SURE"
    ],
    "REPOSITIONING_HELPED": ["YES", "NO", "PARTLY", "NOT_SURE", "NOT_ATTEMPTED"],
    "SKIN_PRESSURE_CONCERN": [
        "NO_CONCERN",
        "REDNESS",
        "DISCOMFORT",
        "SKIN_BREAKDOWN_OR_OPEN_AREA",
        "CAREGIVER_NOT_SURE"
    ],
    "SEATED_DURATION": [
        "LESS_THAN_1_HOUR",
        "1_TO_2_HOURS",
        "2_TO_4_HOURS",
        "MORE_THAN_4_HOURS",
        "NOT_SURE"
    ],
    "BOWEL_BLADDER_ROUTINE_CHANGE": [
        "NO_CHANGE",
        "BOWEL_CHANGE",
        "BLADDER_CHANGE",
        "BOTH",
        "NOT_SURE"
    ],
    "HYDRATION_OR_INTAKE_CONTEXT": [
        "USUAL",
        "LESS_THAN_USUAL",
        "MORE_THAN_USUAL",
        "APPETITE_CHANGE",
        "NOT_SURE"
    ],
    "DISCOMFORT_BOWEL_TIMING": [
        "BEFORE_BATHROOM_ROUTINE",
        "DURING_BATHROOM_ROUTINE",
        "AFTER_BATHROOM_ROUTINE",
        "NOT_RELATED_OR_NOT_SURE"
    ],
    "RESPIRATORY_CONTEXT": [
        "AT_REST",
        "AFTER_ACTIVITY",
        "DURING_SLEEP",
        "DURING_RESPIRATORY_ROUTINE",
        "NOT_SURE"
    ],
    "RESPONSIVENESS_CONTEXT": [
        "LESS_RESPONSIVE_THAN_USUAL",
        "MORE_SLEEPY_THAN_USUAL",
        "HARD_TO_WAKE",
        "RETURNED_TO_BASELINE",
        "NOT_SURE"
    ],
    "SEIZURE_REPORTED_DURATION": [
        "LESS_THAN_1_MIN",
        "1_TO_5_MIN",
        "MORE_THAN_5_MIN",
        "NOT_SURE"
    ],
    "FALL_CONTEXT": [
        "TRANSFER_RELATED",
        "WALKING_OR_STANDING",
        "WHEELCHAIR_OR_SEATING",
        "BATHROOM_RELATED",
        "NOT_SURE"
    ],
    "THERAPY_ROUTINE_STATUS": [
        "COMPLETED_AS_USUAL",
        "COMPLETED_WITH_DIFFICULTY",
        "PARTIALLY_COMPLETED",
        "MISSED",
        "NOT_SURE"
    ],
    "PROVIDER_CONTACT_STATUS": [
        "YES",
        "NO",
        "NOT_INSTRUCTED",
        "NOT_SURE"
    ]
}

with open(OUTPUT_DIR / "uc4_option_sets.json", "w") as f:
    json.dump(OPTION_SETS, f, indent=2)

In [9]:
def field(field_id, label, input_type, options=None, options_source=None,
          maps_to_observation_code=None, maps_to_context_code=None,
          required=False, help_text=None):
    obj = {
        "field_id": field_id,
        "label": label,
        "input_type": input_type,
        "required": required
    }
    if options is not None:
        obj["options"] = options
    if options_source is not None:
        obj["options_source"] = options_source
    if maps_to_observation_code is not None:
        obj["maps_to_observation_code"] = maps_to_observation_code
    if maps_to_context_code is not None:
        obj["maps_to_context_code"] = maps_to_context_code
    if help_text is not None:
        obj["help_text"] = help_text
    return obj

def template(
    template_id,
    title_template,
    priority_type,
    domain,
    activated_by_care_plan_priorities,
    activated_by_template_pack_flags,
    trigger_rules,
    allowed_observation_codes,
    allowed_context_codes,
    what_to_watch,
    what_to_log_next_display,
    what_to_log_next_schema,
    when_to_share_or_escalate,
    safety_boundary,
    safety_flags=None
):
    if safety_flags is None:
        safety_flags = {
            "diagnosis_made": False,
            "medication_change_recommended": False,
            "treatment_change_recommended": False,
            "medication_causality_claimed": False,
            "seizure_detected": False,
            "wound_detected": False,
            "tone_or_spasticity_measured": False
        }
    return {
        "template_id": template_id,
        "schema_version": SCHEMA_VERSION,
        "template_registry_version": TEMPLATE_REGISTRY_VERSION,
        "title_template": title_template,
        "priority_type": priority_type,
        "domain": domain,
        "activated_by_care_plan_priorities": activated_by_care_plan_priorities,
        "activated_by_template_pack_flags": activated_by_template_pack_flags,
        "trigger_rules": trigger_rules,
        "allowed_observation_codes": allowed_observation_codes,
        "allowed_context_codes": allowed_context_codes,
        "what_to_watch": what_to_watch,
        "what_to_log_next_display": what_to_log_next_display,
        "what_to_log_next_schema": what_to_log_next_schema,
        "when_to_share_or_escalate": when_to_share_or_escalate,
        "safety_boundary": safety_boundary,
        "safety_flags": safety_flags
    }

In [11]:
TEMPLATE_REGISTRY = {}

TEMPLATE_REGISTRY["SKIN_PRESSURE_AFTER_SEATED_PERIOD"] = template(
    template_id="SKIN_PRESSURE_AFTER_SEATED_PERIOD",
    title_template="Add a quick skin/pressure check after long seated periods",
    priority_type="blind_spot",
    domain="Skin and positioning support",
    activated_by_care_plan_priorities=["SKIN_PRESSURE_PREVENTION", "POSITIONING_SUPPORT", "MOBILITY_SUPPORT"],
    activated_by_template_pack_flags=["skin_pressure", "mobility_positioning"],
    trigger_rules=["R_HIGH_MOBILITY_SUPPORT", "R_LOW_MOVEMENT_INCREASE", "R_SKIN_LOG_MISSING"],
    allowed_observation_codes=["LOW_MOVEMENT", "SKIN_OR_PRESSURE_CONCERN"],
    allowed_context_codes=["WHILE_SITTING_OR_POSITIONED"],
    what_to_watch=[
        "Redness or discomfort after long seated periods",
        "Areas under pressure",
        "Whether usual repositioning schedule changed"
    ],
    what_to_log_next_display=[
        "Long seated period",
        "Skin/pressure check completed",
        "Any concern noted",
        "Whether repositioning occurred"
    ],
    what_to_log_next_schema=[
        field("long_seated_period", "Was there a long seated period?", "single_select", OPTION_SETS["YES_NO_NOT_SURE"],
              maps_to_observation_code="LOW_MOVEMENT", maps_to_context_code="WHILE_SITTING_OR_POSITIONED"),
        field("seated_duration_estimate", "Approximate seated duration", "single_select", OPTION_SETS["SEATED_DURATION"]),
        field("skin_pressure_check_completed", "Was a skin/pressure check completed?", "single_select", OPTION_SETS["YES_NO_NOT_SURE"]),
        field("skin_pressure_concern_noted", "Any skin or pressure concern noted?", "single_select", OPTION_SETS["SKIN_PRESSURE_CONCERN"],
              maps_to_observation_code="SKIN_OR_PRESSURE_CONCERN"),
        field("repositioning_occurred", "Did repositioning occur?", "single_select", OPTION_SETS["YES_NO_NOT_SURE"])
    ],
    when_to_share_or_escalate="Escalate to the care team for persistent redness, skin breakdown, or caregiver concern.",
    safety_boundary="Care-plan reminder support only; does not detect wounds or diagnose skin issues."
)

TEMPLATE_REGISTRY["MEDICATION_WINDOW_FATIGUE_TRACKING"] = template(
    template_id="MEDICATION_WINDOW_FATIGUE_TRACKING",
    title_template="Track unusual tiredness around medication windows",
    priority_type="recurring_concern",
    domain="Medication-aware observation",
    activated_by_care_plan_priorities=["MEDICATION_ADHERENCE", "SLEEP_ROUTINE_SUPPORT"],
    activated_by_template_pack_flags=["medication_support", "core_caregiver_support"],
    trigger_rules=["R_FATIGUE_RECURRENCE", "R_MED_WATCH_AREA_MATCH", "R_MED_TIMING_CONTEXT_MISSING"],
    allowed_observation_codes=["UNUSUAL_FATIGUE", "MISSED_OR_DELAYED_MEDICATION", "POOR_SLEEP_OR_RESTLESSNESS", "BREATHING_CONCERN", "UNUSUAL_RESPONSIVENESS"],
    allowed_context_codes=["AROUND_MEDICATION_TIME", "DURING_SLEEP_OR_NIGHT", "UNKNOWN_OR_NOT_SURE"],
    what_to_watch=[
        "Whether tiredness is different from usual baseline",
        "Whether it happens before or after scheduled medication times",
        "Whether it appears with poor sleep, low responsiveness, or breathing concern"
    ],
    what_to_log_next_display=[
        "Approximate time tiredness started",
        "Before/after medication window/not sure",
        "Whether a dose was missed or delayed",
        "Whether this differs from baseline"
    ],
    what_to_log_next_schema=[
        field("fatigue_start_time_estimate", "Approximate time tiredness started", "single_select", OPTION_SETS["TIME_OF_DAY"]),
        field("medication_timing_relation", "Timing compared with medication", "single_select", OPTION_SETS["MEDICATION_TIMING_RELATION"],
              maps_to_context_code="AROUND_MEDICATION_TIME"),
        field("dose_missed_or_delayed", "Was a dose missed or delayed?", "single_select", ["NO", "MISSED", "DELAYED", "NOT_SURE"],
              maps_to_observation_code="MISSED_OR_DELAYED_MEDICATION"),
        field("differs_from_baseline", "Is this different from usual baseline?", "single_select", OPTION_SETS["BASELINE_DIFFERENCE"]),
        field("nearby_concerns", "Any nearby concerns?", "multi_select",
              ["POOR_SLEEP_OR_RESTLESSNESS", "UNUSUAL_RESPONSIVENESS", "BREATHING_CONCERN", "APPETITE_OR_HYDRATION_CHANGE", "NONE_NOTED", "NOT_SURE"])
    ],
    when_to_share_or_escalate="Share recurring or worsening fatigue patterns with the care team. Follow emergency plan for severe breathing or responsiveness concerns.",
    safety_boundary="This does not mean medication caused the tiredness and does not recommend medication changes."
)

TEMPLATE_REGISTRY["MISSED_DELAYED_MEDICATION_CONTEXT"] = template(
    template_id="MISSED_DELAYED_MEDICATION_CONTEXT",
    title_template="Log missed or delayed medication with nearby observations",
    priority_type="emerging_pattern",
    domain="Medication routine support",
    activated_by_care_plan_priorities=["MEDICATION_ADHERENCE"],
    activated_by_template_pack_flags=["medication_support"],
    trigger_rules=["R_MISSED_DELAYED_MEDICATION_LOGGED", "R_MEDICATION_ADHERENCE_CARE_PLAN"],
    allowed_observation_codes=["MISSED_OR_DELAYED_MEDICATION", "UNUSUAL_FATIGUE", "POOR_SLEEP_OR_RESTLESSNESS", "PAIN_OR_DISCOMFORT", "UNUSUAL_RESPONSIVENESS"],
    allowed_context_codes=["AROUND_MEDICATION_TIME"],
    what_to_watch=[
        "Unusual fatigue",
        "Sleep/restlessness changes",
        "Unusual responsiveness",
        "Discomfort or behavior changes"
    ],
    what_to_log_next_display=[
        "Which dose was late/missed",
        "Approximate time",
        "Any nearby observation",
        "Whether caregiver contacted provider if instructed by care plan"
    ],
    what_to_log_next_schema=[
        field("medication_identifier", "Which medication or dose?", "medication_select", options_source="patient_medication_profile"),
        field("dose_status", "Dose status", "single_select", OPTION_SETS["DOSE_STATUS"], maps_to_observation_code="MISSED_OR_DELAYED_MEDICATION"),
        field("approximate_time", "Approximate time", "single_select", OPTION_SETS["TIME_OF_DAY"]),
        field("nearby_observations", "Any nearby observations?", "multi_select",
              ["UNUSUAL_FATIGUE", "POOR_SLEEP_OR_RESTLESSNESS", "PAIN_OR_DISCOMFORT", "UNUSUAL_RESPONSIVENESS", "BREATHING_CONCERN", "APPETITE_OR_HYDRATION_CHANGE", "NONE_NOTED", "NOT_SURE"]),
        field("provider_contacted_if_instructed", "Provider contacted if instructed by care plan?", "single_select", OPTION_SETS["PROVIDER_CONTACT_STATUS"])
    ],
    when_to_share_or_escalate="Follow the medication plan or contact the care team if unsure. Do not change or make up doses based on the app.",
    safety_boundary="The app does not recommend medication changes or dosing actions."
)

TEMPLATE_REGISTRY["TRANSFER_DISCOMFORT_TRACKING"] = template(
    template_id="TRANSFER_DISCOMFORT_TRACKING",
    title_template="Track discomfort cues during transfers",
    priority_type="recurring_concern",
    domain="Positioning and movement support",
    activated_by_care_plan_priorities=["MOBILITY_SUPPORT", "POSITIONING_SUPPORT"],
    activated_by_template_pack_flags=["mobility_positioning"],
    trigger_rules=["R_REPEATED_DISCOMFORT", "R_TRANSFER_CONTEXT_CLUSTER", "R_HIGH_MOBILITY_SUPPORT"],
    allowed_observation_codes=["PAIN_OR_DISCOMFORT", "TRANSFER_OR_POSITIONING_CONTEXT"],
    allowed_context_codes=["DURING_TRANSFER"],
    what_to_watch=[
        "Whether discomfort happens before, during, or after transfers",
        "Whether usual repositioning helps",
        "Whether one side or position seems more uncomfortable"
    ],
    what_to_log_next_display=[
        "Transfer timing",
        "Position before and after",
        "Discomfort cues",
        "Whether repositioning helped"
    ],
    what_to_log_next_schema=[
        field("transfer_timing", "Transfer timing", "single_select", OPTION_SETS["TIME_OF_DAY"], maps_to_context_code="DURING_TRANSFER"),
        field("transfer_phase", "When did discomfort happen?", "single_select", OPTION_SETS["TRANSFER_PHASE"]),
        field("position_before", "Position before", "single_select", OPTION_SETS["POSITION"]),
        field("position_after", "Position after", "single_select", OPTION_SETS["POSITION"]),
        field("discomfort_cues", "Discomfort cues", "multi_select", OPTION_SETS["DISCOMFORT_CUES"], maps_to_observation_code="PAIN_OR_DISCOMFORT"),
        field("repositioning_helped", "Did repositioning seem to help?", "single_select", OPTION_SETS["REPOSITIONING_HELPED"])
    ],
    when_to_share_or_escalate="Share with care team if new, worsening, persistent, or associated with distress.",
    safety_boundary="Observation support only; does not diagnose pain source, posture problems, or muscle tone."
)

TEMPLATE_REGISTRY["BOWEL_ROUTINE_DISCOMFORT_CONTEXT"] = template(
    template_id="BOWEL_ROUTINE_DISCOMFORT_CONTEXT",
    title_template="Log bowel routine context when discomfort appears",
    priority_type="emerging_pattern",
    domain="Bowel/bladder routine support",
    activated_by_care_plan_priorities=["BOWEL_ROUTINE_SUPPORT", "HYDRATION_SUPPORT"],
    activated_by_template_pack_flags=["bowel_bladder", "core_caregiver_support"],
    trigger_rules=["R_BOWEL_CHANGE_LOGGED", "R_DISCOMFORT_OVERLAP", "R_BOWEL_WATCH_AREA_MATCH"],
    allowed_observation_codes=["BOWEL_OR_BLADDER_CHANGE", "PAIN_OR_DISCOMFORT", "APPETITE_OR_HYDRATION_CHANGE"],
    allowed_context_codes=["BATHROOM_OR_BOWEL_BLADDER", "MEAL_OR_HYDRATION_RELATED"],
    what_to_watch=[
        "Bowel routine changes",
        "Hydration/intake changes",
        "Whether discomfort occurs near bathroom or bowel/bladder routines"
    ],
    what_to_log_next_display=[
        "Bowel/bladder routine change",
        "Hydration or intake context",
        "Discomfort timing",
        "Whether this differs from usual baseline"
    ],
    what_to_log_next_schema=[
        field("bowel_bladder_routine_change", "Any bowel or bladder routine change?", "single_select", OPTION_SETS["BOWEL_BLADDER_ROUTINE_CHANGE"],
              maps_to_observation_code="BOWEL_OR_BLADDER_CHANGE"),
        field("hydration_or_intake_context", "Hydration or intake context", "single_select", OPTION_SETS["HYDRATION_OR_INTAKE_CONTEXT"],
              maps_to_observation_code="APPETITE_OR_HYDRATION_CHANGE"),
        field("discomfort_timing", "When did discomfort appear?", "single_select", OPTION_SETS["DISCOMFORT_BOWEL_TIMING"]),
        field("differs_from_baseline", "Is this different from usual baseline?", "single_select", OPTION_SETS["BASELINE_DIFFERENCE"])
    ],
    when_to_share_or_escalate="Share persistent bowel routine changes or discomfort patterns with the care team.",
    safety_boundary="Tracking support only; does not diagnose cause or recommend medication changes."
)

TEMPLATE_REGISTRY["BREATHING_CONCERN_CONTEXT"] = template(
    template_id="BREATHING_CONCERN_CONTEXT",
    title_template="Log breathing concern context",
    priority_type="emerging_pattern",
    domain="Respiratory monitoring support",
    activated_by_care_plan_priorities=["RESPIRATORY_MONITORING"],
    activated_by_template_pack_flags=["respiratory_support", "core_caregiver_support"],
    trigger_rules=["R_BREATHING_CONCERN_LOGGED", "R_RESPIRATORY_CARE_PLAN", "R_UC2_CARDIO_RESP_OVERLAP"],
    allowed_observation_codes=["BREATHING_CONCERN", "COLOR_OR_OXYGEN_CONCERN", "UNUSUAL_FATIGUE", "UNUSUAL_RESPONSIVENESS"],
    allowed_context_codes=["RESPIRATORY_ROUTINE_RELATED", "AFTER_ACTIVITY_OR_THERAPY", "DURING_SLEEP_OR_NIGHT"],
    what_to_watch=[
        "Whether breathing concern occurred at rest, after activity, or during sleep",
        "Whether color/oxygen concern was noticed",
        "Whether the emergency plan threshold was crossed"
    ],
    what_to_log_next_display=[
        "Breathing concern context",
        "Any color or oxygen concern",
        "Nearby fatigue or responsiveness change",
        "Whether emergency plan was followed if needed"
    ],
    what_to_log_next_schema=[
        field("breathing_context", "When did the breathing concern occur?", "single_select", OPTION_SETS["RESPIRATORY_CONTEXT"],
              maps_to_observation_code="BREATHING_CONCERN"),
        field("color_or_oxygen_concern", "Any color or oxygen concern?", "single_select", OPTION_SETS["YES_NO_NOT_SURE"],
              maps_to_observation_code="COLOR_OR_OXYGEN_CONCERN"),
        field("nearby_fatigue_or_responsiveness_change", "Nearby fatigue or responsiveness change?", "multi_select",
              ["UNUSUAL_FATIGUE", "UNUSUAL_RESPONSIVENESS", "NONE_NOTED", "NOT_SURE"]),
        field("emergency_plan_followed_if_needed", "Emergency plan followed if needed?", "single_select",
              ["YES", "NO", "NOT_NEEDED", "NOT_SURE"])
    ],
    when_to_share_or_escalate="Follow the emergency plan for severe breathing, color, oxygen, or responsiveness concerns.",
    safety_boundary="Observation support only; does not diagnose respiratory cause or replace emergency thresholds."
)

TEMPLATE_REGISTRY["UNUSUAL_RESPONSIVENESS_CONTEXT"] = template(
    template_id="UNUSUAL_RESPONSIVENESS_CONTEXT",
    title_template="Track unusual responsiveness compared with baseline",
    priority_type="emerging_pattern",
    domain="Responsiveness and baseline-change support",
    activated_by_care_plan_priorities=["PROVIDER_FOLLOWUP", "COMMUNICATION_SUPPORT"],
    activated_by_template_pack_flags=["neuro_support", "core_caregiver_support"],
    trigger_rules=["R_UNUSUAL_RESPONSIVENESS_LOGGED", "R_BASELINE_CHANGE_CONTEXT_NEEDED"],
    allowed_observation_codes=["UNUSUAL_RESPONSIVENESS", "UNUSUAL_FATIGUE", "BREATHING_CONCERN", "SEIZURE_LIKE_EVENT_REPORTED"],
    allowed_context_codes=["UNKNOWN_OR_NOT_SURE", "DURING_SLEEP_OR_NIGHT", "AROUND_MEDICATION_TIME"],
    what_to_watch=[
        "Whether responsiveness is different from usual baseline",
        "Whether it appears with fatigue, breathing concern, or caregiver concern",
        "Whether the emergency plan should be followed"
    ],
    what_to_log_next_display=[
        "What seemed different from baseline",
        "Approximate time",
        "Nearby breathing, fatigue, or medication timing context",
        "Whether baseline returned"
    ],
    what_to_log_next_schema=[
        field("responsiveness_context", "What seemed different?", "multi_select", OPTION_SETS["RESPONSIVENESS_CONTEXT"],
              maps_to_observation_code="UNUSUAL_RESPONSIVENESS"),
        field("approximate_time", "Approximate time", "single_select", OPTION_SETS["TIME_OF_DAY"]),
        field("nearby_context", "Nearby context", "multi_select",
              ["BREATHING_CONCERN", "UNUSUAL_FATIGUE", "AROUND_MEDICATION_TIME", "DURING_SLEEP_OR_NIGHT", "NONE_NOTED", "NOT_SURE"]),
        field("returned_to_baseline", "Did they return to baseline?", "single_select", OPTION_SETS["YES_NO_NOT_SURE"])
    ],
    when_to_share_or_escalate="Follow the emergency plan for severe or persistent responsiveness concerns.",
    safety_boundary="Observation support only; does not diagnose neurologic change."
)

TEMPLATE_REGISTRY["CAREGIVER_REPORTED_SEIZURE_LIKE_EVENT_CONTEXT"] = template(
    template_id="CAREGIVER_REPORTED_SEIZURE_LIKE_EVENT_CONTEXT",
    title_template="Log caregiver-reported seizure-like event context",
    priority_type="emerging_pattern",
    domain="Seizure-action-plan reporting support",
    activated_by_care_plan_priorities=["SEIZURE_ACTION_PLAN_SUPPORT", "PROVIDER_FOLLOWUP"],
    activated_by_template_pack_flags=["seizure_reporting", "neuro_support"],
    trigger_rules=["R_SEIZURE_LIKE_EVENT_REPORTED", "R_SEIZURE_ACTION_PLAN_RELEVANT"],
    allowed_observation_codes=["SEIZURE_LIKE_EVENT_REPORTED", "UNUSUAL_RESPONSIVENESS"],
    allowed_context_codes=["UNKNOWN_OR_NOT_SURE", "DURING_SLEEP_OR_NIGHT"],
    what_to_watch=[
        "Caregiver-reported event timing",
        "Duration estimate",
        "Recovery behavior",
        "Whether seizure action plan was followed"
    ],
    what_to_log_next_display=[
        "Approximate start time",
        "Duration estimate",
        "Recovery/return to baseline",
        "Whether action plan was followed"
    ],
    what_to_log_next_schema=[
        field("event_start_time_estimate", "Approximate start time", "single_select", OPTION_SETS["TIME_OF_DAY"]),
        field("duration_estimate", "Duration estimate", "single_select", OPTION_SETS["SEIZURE_REPORTED_DURATION"]),
        field("returned_to_baseline", "Returned to baseline?", "single_select", OPTION_SETS["YES_NO_NOT_SURE"]),
        field("action_plan_followed", "Seizure action plan followed?", "single_select", ["YES", "NO", "NOT_AVAILABLE", "NOT_SURE"])
    ],
    when_to_share_or_escalate="Follow the existing seizure action plan or emergency plan. Share reported events with the care team.",
    safety_boundary="Reporting support only; does not detect or diagnose seizures.",
    safety_flags={
        "diagnosis_made": False,
        "medication_change_recommended": False,
        "treatment_change_recommended": False,
        "medication_causality_claimed": False,
        "seizure_detected": False,
        "wound_detected": False,
        "tone_or_spasticity_measured": False
    }
)

TEMPLATE_REGISTRY["THERAPY_REHAB_ROUTINE_DIFFICULTY"] = template(
    template_id="THERAPY_REHAB_ROUTINE_DIFFICULTY",
    title_template="Track therapy routine difficulty or plateau",
    priority_type="recurring_concern",
    domain="Therapy and rehabilitation support",
    activated_by_care_plan_priorities=["THERAPY_REHAB_ROUTINE", "MOBILITY_SUPPORT", "PROVIDER_FOLLOWUP"],
    activated_by_template_pack_flags=["therapy_rehab", "mobility_positioning"],
    trigger_rules=["R_THERAPY_ROUTINE_DIFFICULTY_LOGGED", "R_REHAB_TRAJECTORY_CONCERN", "R_PROVIDER_FOLLOWUP_RELEVANT"],
    allowed_observation_codes=["THERAPY_ROUTINE_DIFFICULTY", "PAIN_OR_DISCOMFORT", "LOW_MOVEMENT"],
    allowed_context_codes=["THERAPY_OR_EXERCISE_RELATED", "AFTER_ACTIVITY_OR_THERAPY"],
    what_to_watch=[
        "Whether the exercise was completed as usual",
        "Whether difficulty, fatigue, or discomfort appeared",
        "Whether progress differs from care-plan expectations"
    ],
    what_to_log_next_display=[
        "Therapy routine status",
        "Any discomfort or fatigue",
        "What activity was difficult",
        "Whether caregiver wants provider review"
    ],
    what_to_log_next_schema=[
        field("therapy_routine_status", "Therapy routine status", "single_select", OPTION_SETS["THERAPY_ROUTINE_STATUS"],
              maps_to_observation_code="THERAPY_ROUTINE_DIFFICULTY"),
        field("nearby_observations", "Any discomfort or fatigue?", "multi_select",
              ["PAIN_OR_DISCOMFORT", "UNUSUAL_FATIGUE", "LOW_MOVEMENT", "NONE_NOTED", "NOT_SURE"]),
        field("activity_difficult", "What activity was difficult?", "short_text",
              help_text="Optional short note for provider context; not used for scoring."),
        field("caregiver_wants_provider_review", "Caregiver wants provider review?", "single_select", OPTION_SETS["YES_NO_NOT_SURE"],
              maps_to_observation_code="CAREGIVER_WANTS_PROVIDER_REVIEW")
    ],
    when_to_share_or_escalate="Share recurring difficulty, plateau, or caregiver concern with the therapy or care team.",
    safety_boundary="Rehabilitation tracking support only; does not diagnose decline or prescribe exercises."
)

TEMPLATE_REGISTRY["FALL_OR_NEAR_FALL_CONTEXT"] = template(
    template_id="FALL_OR_NEAR_FALL_CONTEXT",
    title_template="Log fall or near-fall context",
    priority_type="emerging_pattern",
    domain="Safety and mobility support",
    activated_by_care_plan_priorities=["FALL_PREVENTION", "MOBILITY_SUPPORT"],
    activated_by_template_pack_flags=["mobility_positioning", "fall_safety"],
    trigger_rules=["R_FALL_OR_NEAR_FALL_LOGGED", "R_FALL_PREVENTION_RELEVANT"],
    allowed_observation_codes=["FALL_OR_NEAR_FALL", "PAIN_OR_DISCOMFORT", "TRANSFER_OR_POSITIONING_CONTEXT"],
    allowed_context_codes=["DURING_TRANSFER", "WHILE_SITTING_OR_POSITIONED", "UNKNOWN_OR_NOT_SURE"],
    what_to_watch=[
        "Whether event happened during transfer, standing, walking, or seating",
        "Whether pain/discomfort was noticed",
        "Whether caregiver concern or provider review is needed"
    ],
    what_to_log_next_display=[
        "Fall or near-fall context",
        "Any discomfort afterward",
        "Whether assistance/equipment was involved",
        "Whether provider was contacted if needed"
    ],
    what_to_log_next_schema=[
        field("fall_context", "Fall or near-fall context", "single_select", OPTION_SETS["FALL_CONTEXT"],
              maps_to_observation_code="FALL_OR_NEAR_FALL"),
        field("discomfort_afterward", "Any discomfort afterward?", "single_select", OPTION_SETS["YES_NO_NOT_SURE"],
              maps_to_observation_code="PAIN_OR_DISCOMFORT"),
        field("assistance_or_equipment_involved", "Assistance/equipment involved?", "single_select", OPTION_SETS["YES_NO_NOT_SURE"]),
        field("provider_contacted_if_needed", "Provider contacted if needed?", "single_select", OPTION_SETS["PROVIDER_CONTACT_STATUS"])
    ],
    when_to_share_or_escalate="Follow the care plan or emergency plan for injury, distress, or caregiver concern.",
    safety_boundary="Safety logging support only; does not assess injury severity."
)

TEMPLATE_REGISTRY["CAREGIVER_PROVIDER_REVIEW_REQUEST"] = template(
    template_id="CAREGIVER_PROVIDER_REVIEW_REQUEST",
    title_template="Prepare a short pattern summary for provider review",
    priority_type="provider_review",
    domain="Provider communication support",
    activated_by_care_plan_priorities=["PROVIDER_FOLLOWUP"],
    activated_by_template_pack_flags=["core_caregiver_support"],
    trigger_rules=["R_CAREGIVER_WANTS_PROVIDER_REVIEW", "R_RECURRING_CONCERN_MULTIDOMAIN"],
    allowed_observation_codes=["CAREGIVER_WANTS_PROVIDER_REVIEW", "NOT_SURE"],
    allowed_context_codes=["UNKNOWN_OR_NOT_SURE"],
    what_to_watch=[
        "Recurring caregiver concern",
        "Patterns that are new, worsening, or unclear",
        "Questions to share with the care team"
    ],
    what_to_log_next_display=[
        "Main concern",
        "When it happens",
        "What else is nearby",
        "Question for the care team"
    ],
    what_to_log_next_schema=[
        field("main_concern_code", "Main concern", "single_select",
              ["UNUSUAL_FATIGUE", "PAIN_OR_DISCOMFORT", "BREATHING_CONCERN", "BOWEL_OR_BLADDER_CHANGE", "UNUSUAL_RESPONSIVENESS", "OTHER", "NOT_SURE"]),
        field("when_it_happens", "When it happens", "single_select", OPTION_SETS["TIME_OF_DAY"]),
        field("nearby_context", "Nearby context", "multi_select",
              ["AROUND_MEDICATION_TIME", "DURING_TRANSFER", "DURING_SLEEP_OR_NIGHT", "BATHROOM_OR_BOWEL_BLADDER", "AFTER_ACTIVITY_OR_THERAPY", "NOT_SURE"]),
        field("question_for_care_team", "Question for the care team", "short_text",
              help_text="Optional; used for provider summary, not scoring.")
    ],
    when_to_share_or_escalate="Share with the care team when the caregiver wants review or a recurring pattern is unclear.",
    safety_boundary="Communication support only; does not diagnose or recommend treatment."
)

with open(OUTPUT_DIR / "uc4_template_registry.json", "w") as f:
    json.dump(TEMPLATE_REGISTRY, f, indent=2)

len(TEMPLATE_REGISTRY), list(TEMPLATE_REGISTRY.keys())

(11,
 ['SKIN_PRESSURE_AFTER_SEATED_PERIOD',
  'MEDICATION_WINDOW_FATIGUE_TRACKING',
  'MISSED_DELAYED_MEDICATION_CONTEXT',
  'TRANSFER_DISCOMFORT_TRACKING',
  'BOWEL_ROUTINE_DISCOMFORT_CONTEXT',
  'BREATHING_CONCERN_CONTEXT',
  'UNUSUAL_RESPONSIVENESS_CONTEXT',
  'CAREGIVER_REPORTED_SEIZURE_LIKE_EVENT_CONTEXT',
  'THERAPY_REHAB_ROUTINE_DIFFICULTY',
  'FALL_OR_NEAR_FALL_CONTEXT',
  'CAREGIVER_PROVIDER_REVIEW_REQUEST'])

In [13]:
PATIENT_PROFILES = {
    "Mike_DEMO_001": {
        "patient_id": "Mike_DEMO_001",
        "display_name": "Mike",
        "profile_label": "Cerebral palsy, caregiver-supported home routines",
        "primary_condition_context": "cerebral_palsy",
        "mobility_support_level": "HIGH",
        "positioning_support_level": "HIGH",
        "communication_support_level": "MODERATE",
        "baseline_movement_level": "LOW_TO_MODERATE",
        "baseline_fatigue_level": "VARIABLE",
        "skin_pressure_risk_context": "WATCH_AREA",
        "care_plan_priorities": [
            "MOBILITY_SUPPORT",
            "POSITIONING_SUPPORT",
            "SKIN_PRESSURE_PREVENTION",
            "MEDICATION_ADHERENCE",
            "BOWEL_ROUTINE_SUPPORT",
            "COMMUNICATION_SUPPORT",
            "PROVIDER_FOLLOWUP",
            "SEIZURE_ACTION_PLAN_SUPPORT"
        ],
        "template_pack_flags": {
            "core_caregiver_support": True,
            "mobility_positioning": True,
            "skin_pressure": True,
            "medication_support": True,
            "bowel_bladder": True,
            "respiratory_support": False,
            "neuro_support": True,
            "seizure_reporting": True,
            "therapy_rehab": False,
            "fall_safety": True
        },
        "caregiver_priorities": [
            "track_discomfort",
            "track_fatigue",
            "track_positioning",
            "track_medication_timing"
        ]
    },

    "JAMES_DEMO_001": {
        "patient_id": "JAMES_DEMO_001",
        "display_name": "James",
        "profile_label": "Post-stroke rehabilitation",
        "primary_condition_context": "post_stroke_rehabilitation",
        "mobility_support_level": "MODERATE",
        "positioning_support_level": "MODERATE",
        "communication_support_level": "LOW",
        "baseline_movement_level": "IMPROVING_EXPECTED",
        "baseline_fatigue_level": "VARIABLE",
        "skin_pressure_risk_context": "LOW_TO_MODERATE",
        "care_plan_priorities": [
            "THERAPY_REHAB_ROUTINE",
            "MOBILITY_SUPPORT",
            "FALL_PREVENTION",
            "PROVIDER_FOLLOWUP"
        ],
        "template_pack_flags": {
            "core_caregiver_support": True,
            "mobility_positioning": True,
            "skin_pressure": False,
            "medication_support": False,
            "bowel_bladder": False,
            "respiratory_support": False,
            "neuro_support": True,
            "seizure_reporting": False,
            "therapy_rehab": True,
            "fall_safety": True
        },
        "caregiver_priorities": [
            "therapy_progress",
            "mobility_safety",
            "provider_review_if_plateau"
        ]
    },

    "SOFIA_DEMO_001": {
        "patient_id": "SOFIA_DEMO_001",
        "display_name": "Sofia",
        "profile_label": "Spina Bifida with autonomic/GI monitoring context",
        "primary_condition_context": "spina_bifida",
        "mobility_support_level": "MODERATE",
        "positioning_support_level": "MODERATE",
        "communication_support_level": "LOW",
        "baseline_movement_level": "VARIABLE",
        "baseline_fatigue_level": "VARIABLE",
        "skin_pressure_risk_context": "WATCH_AREA",
        "care_plan_priorities": [
            "BOWEL_ROUTINE_SUPPORT",
            "HYDRATION_SUPPORT",
            "SKIN_PRESSURE_PREVENTION",
            "MEDICATION_ADHERENCE",
            "PROVIDER_FOLLOWUP"
        ],
        "template_pack_flags": {
            "core_caregiver_support": True,
            "mobility_positioning": True,
            "skin_pressure": True,
            "medication_support": True,
            "bowel_bladder": True,
            "respiratory_support": False,
            "neuro_support": False,
            "seizure_reporting": False,
            "therapy_rehab": False,
            "fall_safety": False
        },
        "caregiver_priorities": [
            "bowel_bladder_context",
            "hydration_context",
            "fatigue_context"
        ]
    },

    "ELENA_DEMO_001": {
        "patient_id": "ELENA_DEMO_001",
        "display_name": "Elena",
        "profile_label": "COPD with TBI, home-assisted monitoring",
        "primary_condition_context": "copd_tbi",
        "mobility_support_level": "LOW_TO_MODERATE",
        "positioning_support_level": "LOW",
        "communication_support_level": "MODERATE",
        "baseline_movement_level": "LOW_TO_MODERATE",
        "baseline_fatigue_level": "VARIABLE",
        "skin_pressure_risk_context": "LOW",
        "care_plan_priorities": [
            "RESPIRATORY_MONITORING",
            "MEDICATION_ADHERENCE",
            "PROVIDER_FOLLOWUP",
            "SLEEP_ROUTINE_SUPPORT"
        ],
        "template_pack_flags": {
            "core_caregiver_support": True,
            "mobility_positioning": False,
            "skin_pressure": False,
            "medication_support": True,
            "bowel_bladder": False,
            "respiratory_support": True,
            "neuro_support": True,
            "seizure_reporting": False,
            "therapy_rehab": False,
            "fall_safety": True
        },
        "caregiver_priorities": [
            "breathing_context",
            "fatigue_context",
            "responsiveness_context",
            "provider_review_if_recurring"
        ]
    }
}

with open(OUTPUT_DIR / "synthetic_patient_profiles.json", "w") as f:
    json.dump(PATIENT_PROFILES, f, indent=2)

pd.DataFrame(PATIENT_PROFILES.values())

,patient_id,display_name,profile_label,primary_condition_context,mobility_support_level,positioning_support_level,communication_support_level,baseline_movement_level,baseline_fatigue_level,skin_pressure_risk_context,care_plan_priorities,template_pack_flags,caregiver_priorities
0,Mike_DEMO_001,Mike,"Cerebral palsy, caregiver-supported home routines",cerebral_palsy,HIGH,HIGH,MODERATE,LOW_TO_MODERATE,VARIABLE,WATCH_AREA,"[MOBILITY_SUPPORT, POSITIONING_SUPPORT, SKIN_P...","{'core_caregiver_support': True, 'mobility_pos...","[track_discomfort, track_fatigue, track_positi..."
1,JAMES_DEMO_001,James,Post-stroke rehabilitation,post_stroke_rehabilitation,MODERATE,MODERATE,LOW,IMPROVING_EXPECTED,VARIABLE,LOW_TO_MODERATE,"[THERAPY_REHAB_ROUTINE, MOBILITY_SUPPORT, FALL...","{'core_caregiver_support': True, 'mobility_pos...","[therapy_progress, mobility_safety, provider_r..."
2,SOFIA_DEMO_001,Sofia,Spina Bifida with autonomic/GI monitoring context,spina_bifida,MODERATE,MODERATE,LOW,VARIABLE,VARIABLE,WATCH_AREA,"[BOWEL_ROUTINE_SUPPORT, HYDRATION_SUPPORT, SKI...","{'core_caregiver_support': True, 'mobility_pos...","[bowel_bladder_context, hydration_context, fat..."
3,ELENA_DEMO_001,Elena,"COPD with TBI, home-assisted monitoring",copd_tbi,LOW_TO_MODERATE,LOW,MODERATE,LOW_TO_MODERATE,VARIABLE,LOW,"[RESPIRATORY_MONITORING, MEDICATION_ADHERENCE,...","{'core_caregiver_support': True, 'mobility_pos...","[breathing_context, fatigue_context, responsiv..."


In [15]:
MEDICATION_PROFILES = {
    "Mike_DEMO_001": [
        {
            "medication_id": "Mike_med_a",
            "display_name": "Medication A",
            "active": True,
            "schedule": [{"time": "08:00", "window_minutes": 90}, {"time": "20:00", "window_minutes": 90}],
            "watch_area_codes": ["SLEEPINESS_FATIGUE", "WEAKNESS_OR_LOW_TONE_CONCERN"]
        },
        {
            "medication_id": "Mike_med_b",
            "display_name": "Medication B",
            "active": True,
            "schedule": [{"time": "12:00", "window_minutes": 90}],
            "watch_area_codes": ["BOWEL_CHANGE", "APPETITE_OR_HYDRATION_CHANGE"]
        }
    ],
    "JAMES_DEMO_001": [],
    "SOFIA_DEMO_001": [
        {
            "medication_id": "sofia_med_a",
            "display_name": "Medication A",
            "active": True,
            "schedule": [{"time": "09:00", "window_minutes": 90}],
            "watch_area_codes": ["BOWEL_CHANGE", "APPETITE_OR_HYDRATION_CHANGE"]
        }
    ],
    "ELENA_DEMO_001": [
        {
            "medication_id": "elena_med_a",
            "display_name": "Medication A",
            "active": True,
            "schedule": [{"time": "08:00", "window_minutes": 90}, {"time": "20:00", "window_minutes": 90}],
            "watch_area_codes": ["BREATHING_CONCERN", "HEART_RATE_OR_BP_CONCERN"]
        },
        {
            "medication_id": "elena_med_b",
            "display_name": "Medication B",
            "active": True,
            "schedule": [{"time": "22:00", "window_minutes": 90}],
            "watch_area_codes": ["SLEEPINESS_FATIGUE"]
        }
    ]
}

with open(OUTPUT_DIR / "synthetic_medication_profiles.json", "w") as f:
    json.dump(MEDICATION_PROFILES, f, indent=2)

med_rows = []
for pid, meds in MEDICATION_PROFILES.items():
    for med in meds:
        med_rows.append({
            "patient_id": pid,
            "medication_id": med["medication_id"],
            "display_name": med["display_name"],
            "watch_area_codes": "|".join(med["watch_area_codes"]),
            "schedule": json.dumps(med["schedule"])
        })

pd.DataFrame(med_rows).to_csv(OUTPUT_DIR / "synthetic_medication_profiles.csv", index=False)
pd.DataFrame(med_rows)

,patient_id,medication_id,display_name,watch_area_codes,schedule
0,Mike_DEMO_001,Mike_med_a,Medication A,SLEEPINESS_FATIGUE|WEAKNESS_OR_LOW_TONE_CONCERN,"[{""time"": ""08:00"", ""window_minutes"": 90}, {""ti..."
1,Mike_DEMO_001,Mike_med_b,Medication B,BOWEL_CHANGE|APPETITE_OR_HYDRATION_CHANGE,"[{""time"": ""12:00"", ""window_minutes"": 90}]"
2,SOFIA_DEMO_001,sofia_med_a,Medication A,BOWEL_CHANGE|APPETITE_OR_HYDRATION_CHANGE,"[{""time"": ""09:00"", ""window_minutes"": 90}]"
3,ELENA_DEMO_001,elena_med_a,Medication A,BREATHING_CONCERN|HEART_RATE_OR_BP_CONCERN,"[{""time"": ""08:00"", ""window_minutes"": 90}, {""ti..."
4,ELENA_DEMO_001,elena_med_b,Medication B,SLEEPINESS_FATIGUE,"[{""time"": ""22:00"", ""window_minutes"": 90}]"


In [17]:
OBSERVATION_TO_MED_WATCH = {
    "UNUSUAL_FATIGUE": ["SLEEPINESS_FATIGUE"],
    "POOR_SLEEP_OR_RESTLESSNESS": ["SLEEPINESS_FATIGUE", "MOOD_BEHAVIOR_CHANGE"],
    "BREATHING_CONCERN": ["BREATHING_CONCERN"],
    "BOWEL_OR_BLADDER_CHANGE": ["BOWEL_CHANGE"],
    "APPETITE_OR_HYDRATION_CHANGE": ["APPETITE_OR_HYDRATION_CHANGE"],
    "MISSED_OR_DELAYED_MEDICATION": ["MISSED_OR_DELAYED_DOSE"],
    "UNUSUAL_RESPONSIVENESS": ["SLEEPINESS_FATIGUE"]
}

with open(OUTPUT_DIR / "observation_to_medication_watch_area_map.json", "w") as f:
    json.dump(OBSERVATION_TO_MED_WATCH, f, indent=2)

In [20]:
def make_event(patient_id, days_ago, source, observation_codes, context_codes=None,
               severity_hint=1, caregiver_confidence="MEDIUM", uc2_anomaly_type=None,
               uc2_route=None, free_text_note=""):
    if context_codes is None:
        context_codes = ["UNKNOWN_OR_NOT_SURE"]
    return {
        "event_id": f"evt_{patient_id}_{len(EVENTS)+1:04d}",
        "patient_id": patient_id,
        "timestamp": iso(NOW - timedelta(days=days_ago)),
        "source": source,
        "observation_codes": observation_codes,
        "context_codes": context_codes,
        "severity_hint": severity_hint,
        "caregiver_confidence": caregiver_confidence,
        "uc2_anomaly_type": uc2_anomaly_type,
        "uc2_route": uc2_route,
        "free_text_note": free_text_note,
        "free_text_used_for_scoring": False
    }

EVENTS = []

# Mike: fatigue, transfer discomfort, bowel, missed med, no skin logs.
EVENTS += [
    make_event("Mike_DEMO_001", 1, "CAREGIVER_CHECKIN", ["UNUSUAL_FATIGUE"], ["AROUND_MEDICATION_TIME"]),
    make_event("Mike_DEMO_001", 2, "CAREGIVER_CHECKIN", ["UNUSUAL_FATIGUE"], ["UNKNOWN_OR_NOT_SURE"]),
    make_event("Mike_DEMO_001", 3, "CAREGIVER_CHECKIN", ["UNUSUAL_FATIGUE", "POOR_SLEEP_OR_RESTLESSNESS"], ["DURING_SLEEP_OR_NIGHT"]),
    make_event("Mike_DEMO_001", 1, "CAREGIVER_CHECKIN", ["PAIN_OR_DISCOMFORT", "TRANSFER_OR_POSITIONING_CONTEXT"], ["DURING_TRANSFER"]),
    make_event("Mike_DEMO_001", 4, "CAREGIVER_CHECKIN", ["PAIN_OR_DISCOMFORT", "TRANSFER_OR_POSITIONING_CONTEXT"], ["DURING_TRANSFER"]),
    make_event("Mike_DEMO_001", 5, "CAREGIVER_CHECKIN", ["PAIN_OR_DISCOMFORT"], ["WHILE_SITTING_OR_POSITIONED"]),
    make_event("Mike_DEMO_001", 2, "CAREGIVER_CHECKIN", ["BOWEL_OR_BLADDER_CHANGE"], ["BATHROOM_OR_BOWEL_BLADDER"]),
    make_event("Mike_DEMO_001", 1, "CAREGIVER_CHECKIN", ["MISSED_OR_DELAYED_MEDICATION"], ["AROUND_MEDICATION_TIME"]),
]

# James: rehab trajectory/therapy difficulty and fall safety.
EVENTS += [
    make_event("JAMES_DEMO_001", 1, "CAREGIVER_CHECKIN", ["THERAPY_ROUTINE_DIFFICULTY"], ["THERAPY_OR_EXERCISE_RELATED"]),
    make_event("JAMES_DEMO_001", 2, "CAREGIVER_CHECKIN", ["THERAPY_ROUTINE_DIFFICULTY", "UNUSUAL_FATIGUE"], ["AFTER_ACTIVITY_OR_THERAPY"]),
    make_event("JAMES_DEMO_001", 3, "UC3_TRAJECTORY_FAILURE", ["THERAPY_ROUTINE_DIFFICULTY", "CAREGIVER_WANTS_PROVIDER_REVIEW"], ["THERAPY_OR_EXERCISE_RELATED"], severity_hint=2),
    make_event("JAMES_DEMO_001", 5, "CAREGIVER_CHECKIN", ["FALL_OR_NEAR_FALL"], ["DURING_TRANSFER"], severity_hint=2),
]

# Sofia: bowel/bladder, fatigue, hydration, UC2 anomaly.
EVENTS += [
    make_event("SOFIA_DEMO_001", 1, "CAREGIVER_CHECKIN", ["BOWEL_OR_BLADDER_CHANGE", "PAIN_OR_DISCOMFORT"], ["BATHROOM_OR_BOWEL_BLADDER"]),
    make_event("SOFIA_DEMO_001", 2, "CAREGIVER_CHECKIN", ["APPETITE_OR_HYDRATION_CHANGE"], ["MEAL_OR_HYDRATION_RELATED"]),
    make_event("SOFIA_DEMO_001", 3, "UC2_SLOW_PATH_ANOMALY", ["UNUSUAL_FATIGUE"], ["UNKNOWN_OR_NOT_SURE"], severity_hint=2,
               uc2_anomaly_type="CARDIO_RESPIRATORY_SIGNAL_CHANGE", uc2_route="SLM_SUMMARY_AND_PROVIDER_NOTE"),
    make_event("SOFIA_DEMO_001", 4, "CAREGIVER_CHECKIN", ["BOWEL_OR_BLADDER_CHANGE"], ["BATHROOM_OR_BOWEL_BLADDER"]),
]

# Elena: respiratory concerns, fatigue, responsiveness, medication context.
EVENTS += [
    make_event("ELENA_DEMO_001", 1, "CAREGIVER_CHECKIN", ["BREATHING_CONCERN"], ["AFTER_ACTIVITY_OR_THERAPY"], severity_hint=2),
    make_event("ELENA_DEMO_001", 2, "UC2_SLOW_PATH_ANOMALY", ["BREATHING_CONCERN", "UNUSUAL_FATIGUE"], ["UNKNOWN_OR_NOT_SURE"], severity_hint=2,
               uc2_anomaly_type="CARDIO_RESPIRATORY_SIGNAL_CHANGE", uc2_route="SLM_SUMMARY_AND_PROVIDER_NOTE"),
    make_event("ELENA_DEMO_001", 3, "CAREGIVER_CHECKIN", ["UNUSUAL_FATIGUE"], ["AROUND_MEDICATION_TIME"]),
    make_event("ELENA_DEMO_001", 4, "CAREGIVER_CHECKIN", ["UNUSUAL_RESPONSIVENESS"], ["DURING_SLEEP_OR_NIGHT"], severity_hint=2),
]

events_df = pd.DataFrame(EVENTS)
events_df.to_csv(OUTPUT_DIR / "synthetic_shared_care_events.csv", index=False)
events_df

,event_id,patient_id,timestamp,source,observation_codes,context_codes,severity_hint,caregiver_confidence,uc2_anomaly_type,uc2_route,free_text_note,free_text_used_for_scoring
0,evt_Mike_DEMO_001_0001,Mike_DEMO_001,2026-07-15T12:00:00Z,CAREGIVER_CHECKIN,[UNUSUAL_FATIGUE],[AROUND_MEDICATION_TIME],1,MEDIUM,NaN,NaN,,False
1,evt_Mike_DEMO_001_0001,Mike_DEMO_001,2026-07-14T12:00:00Z,CAREGIVER_CHECKIN,[UNUSUAL_FATIGUE],[UNKNOWN_OR_NOT_SURE],1,MEDIUM,NaN,NaN,,False
2,evt_Mike_DEMO_001_0001,Mike_DEMO_001,2026-07-13T12:00:00Z,CAREGIVER_CHECKIN,"[UNUSUAL_FATIGUE, POOR_SLEEP_OR_RESTLESSNESS]",[DURING_SLEEP_OR_NIGHT],1,MEDIUM,NaN,NaN,,False
3,evt_Mike_DEMO_001_0001,Mike_DEMO_001,2026-07-15T12:00:00Z,CAREGIVER_CHECKIN,"[PAIN_OR_DISCOMFORT, TRANSFER_OR_POSITIONING_C...",[DURING_TRANSFER],1,MEDIUM,NaN,NaN,,False
4,evt_Mike_DEMO_001_0001,Mike_DEMO_001,2026-07-12T12:00:00Z,CAREGIVER_CHECKIN,"[PAIN_OR_DISCOMFORT, TRANSFER_OR_POSITIONING_C...",[DURING_TRANSFER],1,MEDIUM,NaN,NaN,,False
5,evt_Mike_DEMO_001_0001,Mike_DEMO_001,2026-07-11T12:00:00Z,CAREGIVER_CHECKIN,[PAIN_OR_DISCOMFORT],[WHILE_SITTING_OR_POSITIONED],1,MEDIUM,NaN,NaN,,False
6,evt_Mike_DEMO_001_0001,Mike_DEMO_001,2026-07-14T12:00:00Z,CAREGIVER_CHECKIN,[BOWEL_OR_BLADDER_CHANGE],[BATHROOM_OR_BOWEL_BLADDER],1,MEDIUM,NaN,NaN,,False
7,evt_Mike_DEMO_001_0001,Mike_DEMO_001,2026-07-15T12:00:00Z,CAREGIVER_CHECKIN,[MISSED_OR_DELAYED_MEDICATION],[AROUND_MEDICATION_TIME],1,MEDIUM,NaN,NaN,,False
8,evt_JAMES_DEMO_001_0009,JAMES_DEMO_001,2026-07-15T12:00:00Z,CAREGIVER_CHECKIN,[THERAPY_ROUTINE_DIFFICULTY],[THERAPY_OR_EXERCISE_RELATED],1,MEDIUM,NaN,NaN,,False
9,evt_JAMES_DEMO_001_0009,JAMES_DEMO_001,2026-07-14T12:00:00Z,CAREGIVER_CHECKIN,"[THERAPY_ROUTINE_DIFFICULTY, UNUSUAL_FATIGUE]",[AFTER_ACTIVITY_OR_THERAPY],1,MEDIUM,NaN,NaN,,False


In [22]:
WEARABLE_SUMMARIES = {
    "Mike_DEMO_001": {
        "low_movement_hours_per_day": 9.5,
        "baseline_low_movement_hours_per_day": 7.0,
        "steps_per_day": 800,
        "baseline_steps_per_day": 1100,
        "respiratory_signal_available": False
    },
    "JAMES_DEMO_001": {
        "low_movement_hours_per_day": 6.5,
        "baseline_low_movement_hours_per_day": 6.0,
        "steps_per_day": 1800,
        "baseline_steps_per_day": 2200,
        "rehab_completion_rate": 0.62,
        "baseline_rehab_completion_rate": 0.85,
        "respiratory_signal_available": False
    },
    "SOFIA_DEMO_001": {
        "low_movement_hours_per_day": 8.0,
        "baseline_low_movement_hours_per_day": 7.5,
        "steps_per_day": 1200,
        "baseline_steps_per_day": 1400,
        "respiratory_signal_available": True
    },
    "ELENA_DEMO_001": {
        "low_movement_hours_per_day": 10.0,
        "baseline_low_movement_hours_per_day": 8.5,
        "steps_per_day": 600,
        "baseline_steps_per_day": 900,
        "respiratory_signal_available": True,
        "spo2_low_events_7d": 1,
        "respiratory_rate_high_events_7d": 1
    }
}

with open(OUTPUT_DIR / "synthetic_wearable_weekly_summaries.json", "w") as f:
    json.dump(WEARABLE_SUMMARIES, f, indent=2)

In [24]:
PREVIOUS_PRIORITIES = {
    "Mike_DEMO_001": ["TRANSFER_DISCOMFORT_TRACKING"],
    "JAMES_DEMO_001": ["THERAPY_REHAB_ROUTINE_DIFFICULTY"],
    "SOFIA_DEMO_001": [],
    "ELENA_DEMO_001": ["BREATHING_CONCERN_CONTEXT"]
}

with open(OUTPUT_DIR / "previous_uc4_priorities.json", "w") as f:
    json.dump(PREVIOUS_PRIORITIES, f, indent=2)

In [26]:
def active_med_watch_areas(patient_id):
    areas = set()
    for med in MEDICATION_PROFILES.get(patient_id, []):
        if med.get("active", False):
            areas.update(med.get("watch_area_codes", []))
    return sorted(areas)

def observation_has_med_watch_match(patient_id, observation_code):
    patient_areas = set(active_med_watch_areas(patient_id))
    mapped = set(OBSERVATION_TO_MED_WATCH.get(observation_code, []))
    return len(patient_areas.intersection(mapped)) > 0

def aggregate_patient(patient_id, lookback_days=7):
    patient_events = [
        e for e in EVENTS
        if e["patient_id"] == patient_id
        and datetime.fromisoformat(e["timestamp"].replace("Z", "+00:00")) >= NOW - timedelta(days=lookback_days)
    ]

    obs_counter = Counter()
    ctx_counter = Counter()
    source_counter = Counter()
    uc2_counter = Counter()

    for e in patient_events:
        source_counter[e["source"]] += 1
        for oc in e["observation_codes"]:
            obs_counter[oc] += 1
        for cc in e["context_codes"]:
            ctx_counter[cc] += 1
        if e.get("uc2_anomaly_type"):
            uc2_counter[e["uc2_anomaly_type"]] += 1

    wearable = WEARABLE_SUMMARIES.get(patient_id, {})
    low_movement_delta = wearable.get("low_movement_hours_per_day", 0) - wearable.get("baseline_low_movement_hours_per_day", 0)

    med_watch_matches = {}
    for oc in OBSERVATION_CODES:
        count = obs_counter.get(oc, 0)
        med_watch_matches[oc] = {
            "count": count,
            "has_med_watch_match": observation_has_med_watch_match(patient_id, oc)
        }

    care_plan = PATIENT_PROFILES[patient_id]["care_plan_priorities"]

    blind_spots = {
        "SKIN_PRESSURE_PREVENTION": (
            "SKIN_PRESSURE_PREVENTION" in care_plan and obs_counter.get("SKIN_OR_PRESSURE_CONCERN", 0) == 0
        ),
        "MEDICATION_ADHERENCE": (
            "MEDICATION_ADHERENCE" in care_plan and obs_counter.get("MISSED_OR_DELAYED_MEDICATION", 0) == 0
        ),
        "BOWEL_ROUTINE_SUPPORT": (
            "BOWEL_ROUTINE_SUPPORT" in care_plan and obs_counter.get("BOWEL_OR_BLADDER_CHANGE", 0) == 0
        ),
        "RESPIRATORY_MONITORING": (
            "RESPIRATORY_MONITORING" in care_plan and obs_counter.get("BREATHING_CONCERN", 0) == 0
        ),
        "THERAPY_REHAB_ROUTINE": (
            "THERAPY_REHAB_ROUTINE" in care_plan and obs_counter.get("THERAPY_ROUTINE_DIFFICULTY", 0) == 0
        )
    }

    return {
        "patient_id": patient_id,
        "lookback_days": lookback_days,
        "event_count": len(patient_events),
        "observation_counts": dict(obs_counter),
        "context_counts": dict(ctx_counter),
        "source_counts": dict(source_counter),
        "uc2_anomaly_counts": dict(uc2_counter),
        "wearable_summary": wearable,
        "low_movement_delta": low_movement_delta,
        "active_medication_watch_areas": active_med_watch_areas(patient_id),
        "med_watch_matches": med_watch_matches,
        "blind_spots": blind_spots,
        "previous_priorities": PREVIOUS_PRIORITIES.get(patient_id, [])
    }

AGGREGATES = {pid: aggregate_patient(pid) for pid in PATIENT_PROFILES}
with open(OUTPUT_DIR / "uc4_patient_aggregates.json", "w") as f:
    json.dump(AGGREGATES, f, indent=2)

AGGREGATES

{'Mike_DEMO_001': {'patient_id': 'Mike_DEMO_001',
  'lookback_days': 7,
  'event_count': 8,
  'observation_counts': {'UNUSUAL_FATIGUE': 3,
   'POOR_SLEEP_OR_RESTLESSNESS': 1,
   'PAIN_OR_DISCOMFORT': 3,
   'TRANSFER_OR_POSITIONING_CONTEXT': 2,
   'BOWEL_OR_BLADDER_CHANGE': 1,
   'MISSED_OR_DELAYED_MEDICATION': 1},
  'context_counts': {'AROUND_MEDICATION_TIME': 2,
   'UNKNOWN_OR_NOT_SURE': 1,
   'DURING_SLEEP_OR_NIGHT': 1,
   'DURING_TRANSFER': 2,
   'WHILE_SITTING_OR_POSITIONED': 1,
   'BATHROOM_OR_BOWEL_BLADDER': 1},
  'source_counts': {'CAREGIVER_CHECKIN': 8},
  'uc2_anomaly_counts': {},
  'wearable_summary': {'low_movement_hours_per_day': 9.5,
   'baseline_low_movement_hours_per_day': 7.0,
   'steps_per_day': 800,
   'baseline_steps_per_day': 1100,
   'respiratory_signal_available': False},
  'low_movement_delta': 2.5,
  'active_medication_watch_areas': ['APPETITE_OR_HYDRATION_CHANGE',
   'BOWEL_CHANGE',
   'SLEEPINESS_FATIGUE',
   'WEAKNESS_OR_LOW_TONE_CONCERN'],
  'med_watch_match

In [28]:
def profile_has_pack(patient_profile, pack):
    return bool(patient_profile.get("template_pack_flags", {}).get(pack, False))

def template_is_activated_for_patient(template_obj, patient_profile):
    care_plan = set(patient_profile["care_plan_priorities"])
    packs = patient_profile.get("template_pack_flags", {})

    care_plan_match = bool(care_plan.intersection(template_obj["activated_by_care_plan_priorities"]))
    pack_match = any(packs.get(pack, False) for pack in template_obj["activated_by_template_pack_flags"])

    return care_plan_match or pack_match

def fire_rules_for_template(template_id, patient_profile, agg):
    obs = defaultdict(int, agg["observation_counts"])
    ctx = defaultdict(int, agg["context_counts"])
    wearable = agg["wearable_summary"]
    care_plan = set(patient_profile["care_plan_priorities"])
    med_watch_areas = set(agg["active_medication_watch_areas"])
    uc2 = defaultdict(int, agg["uc2_anomaly_counts"])

    rules = []
    evidence = {}

    def add(rule, **kwargs):
        rules.append(rule)
        evidence.update(kwargs)

    # Shared conditions
    if patient_profile.get("mobility_support_level") in ["HIGH", "MODERATE"]:
        high_mobility = patient_profile.get("mobility_support_level") == "HIGH"
    else:
        high_mobility = False

    # Template-specific rules
    if template_id == "SKIN_PRESSURE_AFTER_SEATED_PERIOD":
        if patient_profile.get("mobility_support_level") == "HIGH":
            add("R_HIGH_MOBILITY_SUPPORT", mobility_support_level=patient_profile.get("mobility_support_level"))
        if agg["low_movement_delta"] >= 2.0:
            add("R_LOW_MOVEMENT_INCREASE",
                low_movement_hours_per_day=wearable.get("low_movement_hours_per_day"),
                baseline_low_movement_hours_per_day=wearable.get("baseline_low_movement_hours_per_day"))
        if agg["blind_spots"].get("SKIN_PRESSURE_PREVENTION", False):
            add("R_SKIN_LOG_MISSING", skin_pressure_logs_7d=obs["SKIN_OR_PRESSURE_CONCERN"])

    elif template_id == "MEDICATION_WINDOW_FATIGUE_TRACKING":
        if obs["UNUSUAL_FATIGUE"] >= 2:
            add("R_FATIGUE_RECURRENCE", unusual_fatigue_count_7d=obs["UNUSUAL_FATIGUE"])
        if "SLEEPINESS_FATIGUE" in med_watch_areas:
            add("R_MED_WATCH_AREA_MATCH", medication_watch_area="SLEEPINESS_FATIGUE")
        if obs["UNUSUAL_FATIGUE"] >= 2 and ctx["AROUND_MEDICATION_TIME"] < obs["UNUSUAL_FATIGUE"]:
            add("R_MED_TIMING_CONTEXT_MISSING",
                fatigue_count_7d=obs["UNUSUAL_FATIGUE"],
                around_medication_time_context_count_7d=ctx["AROUND_MEDICATION_TIME"])

    elif template_id == "MISSED_DELAYED_MEDICATION_CONTEXT":
        if obs["MISSED_OR_DELAYED_MEDICATION"] >= 1:
            add("R_MISSED_DELAYED_MEDICATION_LOGGED", missed_delayed_medication_count_7d=obs["MISSED_OR_DELAYED_MEDICATION"])
        if "MEDICATION_ADHERENCE" in care_plan:
            add("R_MEDICATION_ADHERENCE_CARE_PLAN", care_plan_priority="MEDICATION_ADHERENCE")

    elif template_id == "TRANSFER_DISCOMFORT_TRACKING":
        if obs["PAIN_OR_DISCOMFORT"] >= 2:
            add("R_REPEATED_DISCOMFORT", pain_discomfort_count_7d=obs["PAIN_OR_DISCOMFORT"])
        if ctx["DURING_TRANSFER"] >= 2 or obs["TRANSFER_OR_POSITIONING_CONTEXT"] >= 2:
            add("R_TRANSFER_CONTEXT_CLUSTER",
                transfer_context_count_7d=obs["TRANSFER_OR_POSITIONING_CONTEXT"],
                during_transfer_context_count_7d=ctx["DURING_TRANSFER"])
        if patient_profile.get("mobility_support_level") == "HIGH":
            add("R_HIGH_MOBILITY_SUPPORT", mobility_support_level=patient_profile.get("mobility_support_level"))

    elif template_id == "BOWEL_ROUTINE_DISCOMFORT_CONTEXT":
        if obs["BOWEL_OR_BLADDER_CHANGE"] >= 1:
            add("R_BOWEL_CHANGE_LOGGED", bowel_bladder_change_count_7d=obs["BOWEL_OR_BLADDER_CHANGE"])
        if obs["PAIN_OR_DISCOMFORT"] >= 1 and obs["BOWEL_OR_BLADDER_CHANGE"] >= 1:
            add("R_DISCOMFORT_OVERLAP",
                pain_discomfort_count_7d=obs["PAIN_OR_DISCOMFORT"],
                bowel_bladder_change_count_7d=obs["BOWEL_OR_BLADDER_CHANGE"])
        if "BOWEL_CHANGE" in med_watch_areas:
            add("R_BOWEL_WATCH_AREA_MATCH", medication_watch_area="BOWEL_CHANGE")

    elif template_id == "BREATHING_CONCERN_CONTEXT":
        if obs["BREATHING_CONCERN"] >= 1:
            add("R_BREATHING_CONCERN_LOGGED", breathing_concern_count_7d=obs["BREATHING_CONCERN"])
        if "RESPIRATORY_MONITORING" in care_plan:
            add("R_RESPIRATORY_CARE_PLAN", care_plan_priority="RESPIRATORY_MONITORING")
        if uc2["CARDIO_RESPIRATORY_SIGNAL_CHANGE"] >= 1:
            add("R_UC2_CARDIO_RESP_OVERLAP", uc2_cardio_respiratory_events_7d=uc2["CARDIO_RESPIRATORY_SIGNAL_CHANGE"])

    elif template_id == "UNUSUAL_RESPONSIVENESS_CONTEXT":
        if obs["UNUSUAL_RESPONSIVENESS"] >= 1:
            add("R_UNUSUAL_RESPONSIVENESS_LOGGED", unusual_responsiveness_count_7d=obs["UNUSUAL_RESPONSIVENESS"])
        if obs["UNUSUAL_RESPONSIVENESS"] >= 1 and ctx["UNKNOWN_OR_NOT_SURE"] >= 1:
            add("R_BASELINE_CHANGE_CONTEXT_NEEDED", unknown_context_count_7d=ctx["UNKNOWN_OR_NOT_SURE"])

    elif template_id == "CAREGIVER_REPORTED_SEIZURE_LIKE_EVENT_CONTEXT":
        if obs["SEIZURE_LIKE_EVENT_REPORTED"] >= 1:
            add("R_SEIZURE_LIKE_EVENT_REPORTED", seizure_like_event_reported_count_7d=obs["SEIZURE_LIKE_EVENT_REPORTED"])
        if "SEIZURE_ACTION_PLAN_SUPPORT" in care_plan:
            add("R_SEIZURE_ACTION_PLAN_RELEVANT", care_plan_priority="SEIZURE_ACTION_PLAN_SUPPORT")

    elif template_id == "THERAPY_REHAB_ROUTINE_DIFFICULTY":
        if obs["THERAPY_ROUTINE_DIFFICULTY"] >= 1:
            add("R_THERAPY_ROUTINE_DIFFICULTY_LOGGED", therapy_difficulty_count_7d=obs["THERAPY_ROUTINE_DIFFICULTY"])
        if wearable.get("rehab_completion_rate", 1.0) < wearable.get("baseline_rehab_completion_rate", 1.0) - 0.15:
            add("R_REHAB_TRAJECTORY_CONCERN",
                rehab_completion_rate=wearable.get("rehab_completion_rate"),
                baseline_rehab_completion_rate=wearable.get("baseline_rehab_completion_rate"))
        if "PROVIDER_FOLLOWUP" in care_plan:
            add("R_PROVIDER_FOLLOWUP_RELEVANT", care_plan_priority="PROVIDER_FOLLOWUP")

    elif template_id == "FALL_OR_NEAR_FALL_CONTEXT":
        if obs["FALL_OR_NEAR_FALL"] >= 1:
            add("R_FALL_OR_NEAR_FALL_LOGGED", fall_or_near_fall_count_7d=obs["FALL_OR_NEAR_FALL"])
        if "FALL_PREVENTION" in care_plan:
            add("R_FALL_PREVENTION_RELEVANT", care_plan_priority="FALL_PREVENTION")

    elif template_id == "CAREGIVER_PROVIDER_REVIEW_REQUEST":
        if obs["CAREGIVER_WANTS_PROVIDER_REVIEW"] >= 1:
            add("R_CAREGIVER_WANTS_PROVIDER_REVIEW", caregiver_provider_review_count_7d=obs["CAREGIVER_WANTS_PROVIDER_REVIEW"])
        recurring_multidomain = sum(1 for k, v in obs.items() if v >= 2 and k not in ["LOOKS_NORMAL", "NOT_SURE"]) >= 2
        if recurring_multidomain:
            add("R_RECURRING_CONCERN_MULTIDOMAIN", recurring_multidomain=True)

    return rules, evidence

In [31]:
def compute_scores(template_id, patient_profile, agg, rules_fired, evidence):
    obs = defaultdict(int, agg["observation_counts"])
    care_plan = set(patient_profile["care_plan_priorities"])
    previous = set(agg["previous_priorities"])

    rule_count = len(rules_fired)
    specificity_score = min(1.0, 0.25 * rule_count + 0.15 * bool(evidence))
    recurrence_score = min(1.0, max(obs.values()) / 4.0 if obs else 0)
    care_plan_relevance_score = 1.0 if set(TEMPLATE_REGISTRY[template_id]["activated_by_care_plan_priorities"]).intersection(care_plan) else 0.4
    novelty_score = 0.7 if template_id not in previous else 0.35
    repeat_penalty = 0.15 if template_id in previous else 0.0

    blind_spot_score = 0.0
    if template_id == "SKIN_PRESSURE_AFTER_SEATED_PERIOD" and agg["blind_spots"].get("SKIN_PRESSURE_PREVENTION"):
        blind_spot_score = 1.0
    elif any(agg["blind_spots"].values()):
        blind_spot_score = 0.25

    caregiver_priority_score = 0.5
    caregiver_priorities = patient_profile.get("caregiver_priorities", [])
    if template_id == "TRANSFER_DISCOMFORT_TRACKING" and "track_discomfort" in caregiver_priorities:
        caregiver_priority_score = 1.0
    elif template_id == "MEDICATION_WINDOW_FATIGUE_TRACKING" and "track_fatigue" in caregiver_priorities:
        caregiver_priority_score = 1.0
    elif template_id == "BREATHING_CONCERN_CONTEXT" and "breathing_context" in caregiver_priorities:
        caregiver_priority_score = 1.0
    elif template_id == "THERAPY_REHAB_ROUTINE_DIFFICULTY" and "therapy_progress" in caregiver_priorities:
        caregiver_priority_score = 1.0

    burden_penalty = 0.05 * len(TEMPLATE_REGISTRY[template_id]["what_to_log_next_schema"])
    burden_penalty = min(0.35, burden_penalty)

    final_score = (
        0.25 * specificity_score
        + 0.20 * recurrence_score
        + 0.20 * care_plan_relevance_score
        + 0.15 * novelty_score
        + 0.10 * blind_spot_score
        + 0.10 * caregiver_priority_score
        - 0.10 * burden_penalty
        - 0.10 * repeat_penalty
    )

    return {
        "specificity_score": round(specificity_score, 3),
        "recurrence_score": round(recurrence_score, 3),
        "care_plan_relevance_score": round(care_plan_relevance_score, 3),
        "novelty_score": round(novelty_score, 3),
        "blind_spot_score": round(blind_spot_score, 3),
        "caregiver_priority_score": round(caregiver_priority_score, 3),
        "burden_penalty": round(burden_penalty, 3),
        "repeat_penalty": round(repeat_penalty, 3),
        "final_score": round(max(0.0, min(1.0, final_score)), 3)
    }

In [34]:
def generate_candidates_for_patient(patient_id):
    patient_profile = PATIENT_PROFILES[patient_id]
    agg = AGGREGATES[patient_id]

    candidates = []

    for tid, tmpl in TEMPLATE_REGISTRY.items():
        if not template_is_activated_for_patient(tmpl, patient_profile):
            continue

        rules_fired, evidence = fire_rules_for_template(tid, patient_profile, agg)

        # Require at least one fired rule to generate a candidate.
        if not rules_fired:
            continue

        scores = compute_scores(tid, patient_profile, agg, rules_fired, evidence)

        candidates.append({
            "patient_id": patient_id,
            "template_id": tid,
            "rules_fired": rules_fired,
            "evidence": evidence,
            "scores": scores
        })

    return sorted(candidates, key=lambda x: x["scores"]["final_score"], reverse=True)

ALL_CANDIDATES = {pid: generate_candidates_for_patient(pid) for pid in PATIENT_PROFILES}

candidate_rows = []
for pid, candidates in ALL_CANDIDATES.items():
    for c in candidates:
        candidate_rows.append({
            "patient_id": pid,
            "template_id": c["template_id"],
            "rules_fired": "|".join(c["rules_fired"]),
            "final_score": c["scores"]["final_score"]
        })

candidates_df = pd.DataFrame(candidate_rows)
candidates_df.to_csv(OUTPUT_DIR / "uc4_micro_priority_candidates.csv", index=False)
candidates_df

,patient_id,template_id,rules_fired,final_score
0,Mike_DEMO_001,SKIN_PRESSURE_AFTER_SEATED_PERIOD,R_HIGH_MOBILITY_SUPPORT|R_LOW_MOVEMENT_INCREAS...,0.805
1,Mike_DEMO_001,MEDICATION_WINDOW_FATIGUE_TRACKING,R_FATIGUE_RECURRENCE|R_MED_WATCH_AREA_MATCH|R_...,0.780
2,Mike_DEMO_001,BOWEL_ROUTINE_DISCOMFORT_CONTEXT,R_BOWEL_CHANGE_LOGGED|R_DISCOMFORT_OVERLAP|R_B...,0.735
3,Mike_DEMO_001,TRANSFER_DISCOMFORT_TRACKING,R_REPEATED_DISCOMFORT|R_TRANSFER_CONTEXT_CLUST...,0.707
4,Mike_DEMO_001,MISSED_DELAYED_MEDICATION_CONTEXT,R_MISSED_DELAYED_MEDICATION_LOGGED|R_MEDICATIO...,0.667
5,Mike_DEMO_001,CAREGIVER_REPORTED_SEIZURE_LIKE_EVENT_CONTEXT,R_SEIZURE_ACTION_PLAN_RELEVANT,0.610
6,Mike_DEMO_001,THERAPY_REHAB_ROUTINE_DIFFICULTY,R_PROVIDER_FOLLOWUP_RELEVANT,0.610
7,Mike_DEMO_001,CAREGIVER_PROVIDER_REVIEW_REQUEST,R_RECURRING_CONCERN_MULTIDOMAIN,0.610
8,JAMES_DEMO_001,THERAPY_REHAB_ROUTINE_DIFFICULTY,R_THERAPY_ROUTINE_DIFFICULTY_LOGGED|R_REHAB_TR...,0.692
9,JAMES_DEMO_001,FALL_OR_NEAR_FALL_CONTEXT,R_FALL_OR_NEAR_FALL_LOGGED|R_FALL_PREVENTION_R...,0.647


In [36]:
def select_top_priorities(candidates, max_items=5):
    selected = []
    domain_counts = Counter()

    for c in candidates:
        domain = TEMPLATE_REGISTRY[c["template_id"]]["domain"]

        # Simple burden/diversity caps
        if domain_counts[domain] >= 2:
            continue

        selected.append(c)
        domain_counts[domain] += 1

        if len(selected) >= max_items:
            break

    return selected

TOP_PRIORITIES_RAW = {
    pid: select_top_priorities(candidates, max_items=5)
    for pid, candidates in ALL_CANDIDATES.items()
}

In [38]:
def render_why_lines(template_id, evidence, patient_profile):
    name = patient_profile["display_name"]
    lines = []

    for k, v in evidence.items():
        if k == "mobility_support_level":
            lines.append(f"Patient profile indicates {str(v).lower()} mobility support.")
        elif k == "low_movement_hours_per_day":
            baseline = evidence.get("baseline_low_movement_hours_per_day", None)
            if baseline is not None:
                lines.append(f"Low-movement hours/day: {v}. Baseline: {baseline}.")
        elif k == "skin_pressure_logs_7d":
            lines.append(f"No skin/pressure concern logs were recorded in the past 7 days.")
        elif k == "unusual_fatigue_count_7d":
            lines.append(f"Unusual fatigue selected {v} times in 7 days.")
        elif k == "medication_watch_area":
            if v == "SLEEPINESS_FATIGUE":
                lines.append("Medication profile includes sleepiness/fatigue as a watch area.")
            elif v == "BOWEL_CHANGE":
                lines.append("Medication profile includes bowel routine as a watch area.")
            else:
                lines.append(f"Medication profile includes {v} as a watch area.")
        elif k == "missed_delayed_medication_count_7d":
            lines.append(f"Missed/delayed medication count: {v}.")
        elif k == "pain_discomfort_count_7d":
            lines.append(f"Pain/discomfort selected {v} times in 7 days.")
        elif k == "transfer_context_count_7d":
            lines.append(f"Transfer context selected {v} times in 7 days.")
        elif k == "during_transfer_context_count_7d":
            if v > 0:
                lines.append(f"During-transfer context selected {v} times in 7 days.")
        elif k == "bowel_bladder_change_count_7d":
            lines.append(f"Bowel/bladder change count: {v}.")
        elif k == "breathing_concern_count_7d":
            lines.append(f"Breathing concern selected {v} times in 7 days.")
        elif k == "uc2_cardio_respiratory_events_7d":
            lines.append(f"UC2 cardio-respiratory signal change appeared {v} time(s) in 7 days.")
        elif k == "unusual_responsiveness_count_7d":
            lines.append(f"Unusual responsiveness selected {v} time(s) in 7 days.")
        elif k == "therapy_difficulty_count_7d":
            lines.append(f"Therapy routine difficulty selected {v} time(s) in 7 days.")
        elif k == "rehab_completion_rate":
            baseline = evidence.get("baseline_rehab_completion_rate", None)
            if baseline is not None:
                lines.append(f"Therapy completion rate: {v}. Baseline: {baseline}.")
        elif k == "fall_or_near_fall_count_7d":
            lines.append(f"Fall or near-fall logged {v} time(s) in 7 days.")
        elif k == "care_plan_priority":
            lines.append(f"{v.replace('_', ' ').title()} is listed as a care-plan priority.")

    # Safety-specific medication line
    if template_id in ["MEDICATION_WINDOW_FATIGUE_TRACKING", "BOWEL_ROUTINE_DISCOMFORT_CONTEXT"]:
        if any("Medication profile includes" in line for line in lines):
            lines.append("Medication cause is not inferred.")

    if not lines:
        lines.append("Structured care-plan and recent observation data matched this priority.")

    return lines

def render_priority_card(patient_id, candidate, rank):
    patient_profile = PATIENT_PROFILES[patient_id]
    tmpl = TEMPLATE_REGISTRY[candidate["template_id"]]
    evidence = candidate["evidence"]

    priority_id = f"uc4_priority_{patient_id}_{rank:02d}_{candidate['template_id']}"

    card = {
        "priority_id": priority_id,
        "template_id": candidate["template_id"],
        "schema_version": SCHEMA_VERSION,
        "template_registry_version": TEMPLATE_REGISTRY_VERSION,
        "engine_version": ENGINE_VERSION,
        "patient_id": patient_id,
        "patient_display_name": patient_profile["display_name"],
        "rank": rank,
        "priority_type": tmpl["priority_type"],
        "domain": tmpl["domain"],
        "title": tmpl["title_template"],
        "why_this_is_on_the_list": render_why_lines(candidate["template_id"], evidence, patient_profile),
        "what_to_watch": tmpl["what_to_watch"],
        "what_to_log_next_display": tmpl["what_to_log_next_display"],
        "what_to_log_next_schema": tmpl["what_to_log_next_schema"],
        "when_to_share_or_escalate": tmpl["when_to_share_or_escalate"],
        "safety_boundary": tmpl["safety_boundary"],
        "scores": candidate["scores"],
        "evidence": {
            "rules_fired": candidate["rules_fired"],
            "structured_evidence": evidence,
            "lookback_days": AGGREGATES[patient_id]["lookback_days"],
            "free_text_used_for_scoring": False
        },
        "safety_flags": tmpl["safety_flags"],
        "audit": {
            "generated_by": "UC4_RULE_TEMPLATE_ENGINE",
            "slm_used": False,
            "free_text_used_for_scoring": False,
            "template_registry_version": TEMPLATE_REGISTRY_VERSION,
            "schema_version": SCHEMA_VERSION,
            "created_at": iso(NOW)
        }
    }
    return card

PRIORITY_CARDS = {}
for pid, candidates in TOP_PRIORITIES_RAW.items():
    PRIORITY_CARDS[pid] = [
        render_priority_card(pid, c, rank=i+1)
        for i, c in enumerate(candidates)
    ]

with open(OUTPUT_DIR / "uc4_structured_priority_cards_by_patient.json", "w") as f:
    json.dump(PRIORITY_CARDS, f, indent=2)

PRIORITY_CARDS

{'Mike_DEMO_001': [{'priority_id': 'uc4_priority_Mike_DEMO_001_01_SKIN_PRESSURE_AFTER_SEATED_PERIOD',
   'template_id': 'SKIN_PRESSURE_AFTER_SEATED_PERIOD',
   'schema_version': 'uc4_schema_v0.1.0',
   'template_registry_version': 'uc4_template_registry_v0.1.0',
   'engine_version': 'uc4_structured_micropriority_engine_v0.1.0',
   'patient_id': 'Mike_DEMO_001',
   'patient_display_name': 'Mike',
   'rank': 1,
   'priority_type': 'blind_spot',
   'domain': 'Skin and positioning support',
   'title': 'Add a quick skin/pressure check after long seated periods',
   'why_this_is_on_the_list': ['Patient profile indicates high mobility support.',
    'Low-movement hours/day: 9.5. Baseline: 7.0.',
    'No skin/pressure concern logs were recorded in the past 7 days.'],
   'what_to_watch': ['Redness or discomfort after long seated periods',
    'Areas under pressure',
    'Whether usual repositioning schedule changed'],
   'what_to_log_next_display': ['Long seated period',
    'Skin/pressure che

In [40]:
flat_cards = []
for pid, cards in PRIORITY_CARDS.items():
    for card in cards:
        flat_cards.append({
            "patient_id": pid,
            "patient": card["patient_display_name"],
            "rank": card["rank"],
            "template_id": card["template_id"],
            "title": card["title"],
            "domain": card["domain"],
            "final_score": card["scores"]["final_score"],
            "rules_fired": "|".join(card["evidence"]["rules_fired"]),
            "schema_field_count": len(card["what_to_log_next_schema"])
        })

top_cards_df = pd.DataFrame(flat_cards)
top_cards_df.to_csv(OUTPUT_DIR / "uc4_top_structured_priorities.csv", index=False)
top_cards_df

,patient_id,patient,rank,template_id,title,domain,final_score,rules_fired,schema_field_count
0,Mike_DEMO_001,Mike,1,SKIN_PRESSURE_AFTER_SEATED_PERIOD,Add a quick skin/pressure check after long sea...,Skin and positioning support,0.805,R_HIGH_MOBILITY_SUPPORT|R_LOW_MOVEMENT_INCREAS...,5
1,Mike_DEMO_001,Mike,2,MEDICATION_WINDOW_FATIGUE_TRACKING,Track unusual tiredness around medication windows,Medication-aware observation,0.780,R_FATIGUE_RECURRENCE|R_MED_WATCH_AREA_MATCH|R_...,5
2,Mike_DEMO_001,Mike,3,BOWEL_ROUTINE_DISCOMFORT_CONTEXT,Log bowel routine context when discomfort appears,Bowel/bladder routine support,0.735,R_BOWEL_CHANGE_LOGGED|R_DISCOMFORT_OVERLAP|R_B...,4
3,Mike_DEMO_001,Mike,4,TRANSFER_DISCOMFORT_TRACKING,Track discomfort cues during transfers,Positioning and movement support,0.707,R_REPEATED_DISCOMFORT|R_TRANSFER_CONTEXT_CLUST...,6
4,Mike_DEMO_001,Mike,5,MISSED_DELAYED_MEDICATION_CONTEXT,Log missed or delayed medication with nearby o...,Medication routine support,0.667,R_MISSED_DELAYED_MEDICATION_LOGGED|R_MEDICATIO...,5
5,JAMES_DEMO_001,James,1,THERAPY_REHAB_ROUTINE_DIFFICULTY,Track therapy routine difficulty or plateau,Therapy and rehabilitation support,0.692,R_THERAPY_ROUTINE_DIFFICULTY_LOGGED|R_REHAB_TR...,4
6,JAMES_DEMO_001,James,2,FALL_OR_NEAR_FALL_CONTEXT,Log fall or near-fall context,Safety and mobility support,0.647,R_FALL_OR_NEAR_FALL_LOGGED|R_FALL_PREVENTION_R...,4
7,JAMES_DEMO_001,James,3,CAREGIVER_PROVIDER_REVIEW_REQUEST,Prepare a short pattern summary for provider r...,Provider communication support,0.585,R_CAREGIVER_WANTS_PROVIDER_REVIEW,4
8,SOFIA_DEMO_001,Sofia,1,BOWEL_ROUTINE_DISCOMFORT_CONTEXT,Log bowel routine context when discomfort appears,Bowel/bladder routine support,0.685,R_BOWEL_CHANGE_LOGGED|R_DISCOMFORT_OVERLAP|R_B...,4
9,SOFIA_DEMO_001,Sofia,2,SKIN_PRESSURE_AFTER_SEATED_PERIOD,Add a quick skin/pressure check after long sea...,Skin and positioning support,0.630,R_SKIN_LOG_MISSING,5


In [42]:
def print_card(card):
    print("=" * 100)
    print(f"{card['patient_display_name']} | Priority {card['rank']}: {card['title']}")
    print()
    print("Why this is on the list:")
    for line in card["why_this_is_on_the_list"]:
        print(f"- {line}")
    print()
    print("What to watch:")
    for line in card["what_to_watch"]:
        print(f"- {line}")
    print()
    print("What to log next:")
    for line in card["what_to_log_next_display"]:
        print(f"- {line}")
    print()
    print("Structured schema fields:")
    for f in card["what_to_log_next_schema"]:
        print(f"- {f['field_id']} | {f['input_type']} | {f.get('options', f.get('options_source', ''))}")
    print()
    print("When to share or escalate:")
    print(card["when_to_share_or_escalate"])
    print()
    print("Safety boundary:")
    print(card["safety_boundary"])
    print()

for pid, cards in PRIORITY_CARDS.items():
    for card in cards:
        print_card(card)

Mike | Priority 1: Add a quick skin/pressure check after long seated periods

Why this is on the list:
- Patient profile indicates high mobility support.
- Low-movement hours/day: 9.5. Baseline: 7.0.
- No skin/pressure concern logs were recorded in the past 7 days.

What to watch:
- Redness or discomfort after long seated periods
- Areas under pressure
- Whether usual repositioning schedule changed

What to log next:
- Long seated period
- Skin/pressure check completed
- Any concern noted
- Whether repositioning occurred

Structured schema fields:
- long_seated_period | single_select | ['YES', 'NO', 'NOT_SURE']
- seated_duration_estimate | single_select | ['LESS_THAN_1_HOUR', '1_TO_2_HOURS', '2_TO_4_HOURS', 'MORE_THAN_4_HOURS', 'NOT_SURE']
- skin_pressure_check_completed | single_select | ['YES', 'NO', 'NOT_SURE']
- skin_pressure_concern_noted | single_select | ['NO_CONCERN', 'REDNESS', 'DISCOMFORT', 'SKIN_BREAKDOWN_OR_OPEN_AREA', 'CAREGIVER_NOT_SURE']
- repositioning_occurred | single

In [44]:
def example_response_for_field(f):
    input_type = f["input_type"]

    if input_type == "single_select":
        options = f.get("options", ["NOT_SURE"])
        # Prefer meaningful non-risky choices for examples
        for preferred in ["YES", "PARTLY", "EVENING", "DURING_TRANSFER", "YES_DIFFERENT", "NOT_SURE"]:
            if preferred in options:
                return preferred
        return options[0]

    if input_type == "multi_select":
        options = f.get("options", ["NOT_SURE"])
        safe = [o for o in options if o not in ["SKIN_BREAKDOWN_OR_OPEN_AREA"]]
        return safe[:2] if len(safe) >= 2 else safe

    if input_type == "medication_select":
        return "example_medication_id_from_patient_profile"

    if input_type == "short_text":
        return "Optional caregiver note for provider context."

    if input_type == "boolean":
        return True

    if input_type == "number":
        return 1

    if input_type == "time":
        return "18:00"

    return None

def build_example_caregiver_response(card):
    responses = {}
    observation_codes = set()
    context_codes = set()

    for f in card["what_to_log_next_schema"]:
        responses[f["field_id"]] = example_response_for_field(f)
        if "maps_to_observation_code" in f:
            observation_codes.add(f["maps_to_observation_code"])
        if "maps_to_context_code" in f:
            context_codes.add(f["maps_to_context_code"])

    if not observation_codes:
        observation_codes.update(TEMPLATE_REGISTRY[card["template_id"]]["allowed_observation_codes"][:1])
    if not context_codes:
        allowed_ctx = TEMPLATE_REGISTRY[card["template_id"]]["allowed_context_codes"]
        context_codes.update(allowed_ctx[:1] if allowed_ctx else ["UNKNOWN_OR_NOT_SURE"])

    return {
        "log_id": f"uc4_log_response_{card['priority_id']}",
        "patient_id": card["patient_id"],
        "source_priority_id": card["priority_id"],
        "template_id": card["template_id"],
        "timestamp": iso(NOW + timedelta(hours=6)),
        "event_source": "CAREGIVER_UC4_CHECKLIST",
        "observation_codes": sorted(observation_codes),
        "context_codes": sorted(context_codes),
        "responses": responses,
        "free_text_note": "",
        "free_text_used_for_scoring": False,
        "caregiver_confidence": "MEDIUM",
        "audit": {
            "schema_version": SCHEMA_VERSION,
            "template_registry_version": TEMPLATE_REGISTRY_VERSION,
            "created_by": "caregiver",
            "consent_checked": True,
            "slm_used": False
        }
    }

CAREGIVER_RESPONSE_EXAMPLES = {}
for pid, cards in PRIORITY_CARDS.items():
    CAREGIVER_RESPONSE_EXAMPLES[pid] = [
        build_example_caregiver_response(card)
        for card in cards
    ]

with open(OUTPUT_DIR / "uc4_caregiver_response_examples.json", "w") as f:
    json.dump(CAREGIVER_RESPONSE_EXAMPLES, f, indent=2)

CAREGIVER_RESPONSE_EXAMPLES

{'Mike_DEMO_001': [{'log_id': 'uc4_log_response_uc4_priority_Mike_DEMO_001_01_SKIN_PRESSURE_AFTER_SEATED_PERIOD',
   'patient_id': 'Mike_DEMO_001',
   'source_priority_id': 'uc4_priority_Mike_DEMO_001_01_SKIN_PRESSURE_AFTER_SEATED_PERIOD',
   'template_id': 'SKIN_PRESSURE_AFTER_SEATED_PERIOD',
   'timestamp': '2026-07-16T18:00:00Z',
   'event_source': 'CAREGIVER_UC4_CHECKLIST',
   'observation_codes': ['LOW_MOVEMENT', 'SKIN_OR_PRESSURE_CONCERN'],
   'context_codes': ['WHILE_SITTING_OR_POSITIONED'],
   'responses': {'long_seated_period': 'YES',
    'seated_duration_estimate': 'NOT_SURE',
    'skin_pressure_check_completed': 'YES',
    'skin_pressure_concern_noted': 'NO_CONCERN',
    'repositioning_occurred': 'YES'},
   'free_text_note': '',
   'free_text_used_for_scoring': False,
   'caregiver_confidence': 'MEDIUM',
   'audit': {'schema_version': 'uc4_schema_v0.1.0',
    'template_registry_version': 'uc4_template_registry_v0.1.0',
    'created_by': 'caregiver',
    'consent_checked': Tr

In [46]:
def append_caregiver_responses_to_events(response_examples):
    new_events = []

    for pid, responses in response_examples.items():
        for r in responses:
            new_events.append({
                "event_id": r["log_id"],
                "patient_id": r["patient_id"],
                "timestamp": r["timestamp"],
                "source": "CAREGIVER_UC4_CHECKLIST",
                "observation_codes": r["observation_codes"],
                "context_codes": r["context_codes"],
                "severity_hint": 1,
                "caregiver_confidence": r["caregiver_confidence"],
                "uc2_anomaly_type": None,
                "uc2_route": None,
                "free_text_note": r.get("free_text_note", ""),
                "free_text_used_for_scoring": False,
                "source_priority_id": r["source_priority_id"],
                "template_id": r["template_id"],
                "responses": r["responses"]
            })

    return new_events

NEXT_CYCLE_RESPONSE_EVENTS = append_caregiver_responses_to_events(CAREGIVER_RESPONSE_EXAMPLES)

with open(OUTPUT_DIR / "uc4_next_cycle_response_events.json", "w") as f:
    json.dump(NEXT_CYCLE_RESPONSE_EVENTS, f, indent=2)

pd.DataFrame(NEXT_CYCLE_RESPONSE_EVENTS).head()

,event_id,patient_id,timestamp,source,observation_codes,context_codes,severity_hint,caregiver_confidence,uc2_anomaly_type,uc2_route,free_text_note,free_text_used_for_scoring,source_priority_id,template_id,responses
0,uc4_log_response_uc4_priority_Mike_DEMO_001_01...,Mike_DEMO_001,2026-07-16T18:00:00Z,CAREGIVER_UC4_CHECKLIST,"[LOW_MOVEMENT, SKIN_OR_PRESSURE_CONCERN]",[WHILE_SITTING_OR_POSITIONED],1,MEDIUM,None,None,,False,uc4_priority_Mike_DEMO_001_01_SKIN_PRESSURE_AF...,SKIN_PRESSURE_AFTER_SEATED_PERIOD,"{'long_seated_period': 'YES', 'seated_duration..."
1,uc4_log_response_uc4_priority_Mike_DEMO_001_02...,Mike_DEMO_001,2026-07-16T18:00:00Z,CAREGIVER_UC4_CHECKLIST,[MISSED_OR_DELAYED_MEDICATION],[AROUND_MEDICATION_TIME],1,MEDIUM,None,None,,False,uc4_priority_Mike_DEMO_001_02_MEDICATION_WINDO...,MEDICATION_WINDOW_FATIGUE_TRACKING,"{'fatigue_start_time_estimate': 'EVENING', 'me..."
2,uc4_log_response_uc4_priority_Mike_DEMO_001_03...,Mike_DEMO_001,2026-07-16T18:00:00Z,CAREGIVER_UC4_CHECKLIST,"[APPETITE_OR_HYDRATION_CHANGE, BOWEL_OR_BLADDE...",[BATHROOM_OR_BOWEL_BLADDER],1,MEDIUM,None,None,,False,uc4_priority_Mike_DEMO_001_03_BOWEL_ROUTINE_DI...,BOWEL_ROUTINE_DISCOMFORT_CONTEXT,"{'bowel_bladder_routine_change': 'NOT_SURE', '..."
3,uc4_log_response_uc4_priority_Mike_DEMO_001_04...,Mike_DEMO_001,2026-07-16T18:00:00Z,CAREGIVER_UC4_CHECKLIST,[PAIN_OR_DISCOMFORT],[DURING_TRANSFER],1,MEDIUM,None,None,,False,uc4_priority_Mike_DEMO_001_04_TRANSFER_DISCOMF...,TRANSFER_DISCOMFORT_TRACKING,"{'transfer_timing': 'EVENING', 'transfer_phase..."
4,uc4_log_response_uc4_priority_Mike_DEMO_001_05...,Mike_DEMO_001,2026-07-16T18:00:00Z,CAREGIVER_UC4_CHECKLIST,[MISSED_OR_DELAYED_MEDICATION],[AROUND_MEDICATION_TIME],1,MEDIUM,None,None,,False,uc4_priority_Mike_DEMO_001_05_MISSED_DELAYED_M...,MISSED_DELAYED_MEDICATION_CONTEXT,{'medication_identifier': 'example_medication_...


In [48]:
def derive_features_from_responses(patient_id, response_events):
    patient_responses = [e for e in response_events if e["patient_id"] == patient_id]

    template_counter = Counter()
    field_value_counter = defaultdict(Counter)

    for e in patient_responses:
        template_counter[e["template_id"]] += 1
        for field_id, value in e.get("responses", {}).items():
            if isinstance(value, list):
                for item in value:
                    field_value_counter[field_id][item] += 1
            else:
                field_value_counter[field_id][value] += 1

    derived = {
        "patient_id": patient_id,
        "response_event_count": len(patient_responses),
        "template_response_counts": dict(template_counter),
        "field_value_counts": {
            k: dict(v)
            for k, v in field_value_counter.items()
        },
        "free_text_used_for_scoring": False
    }

    # Example derived metrics useful next cycle
    repositioning_counts = field_value_counter.get("repositioning_helped", Counter())
    total_repositioning = sum(repositioning_counts.values())
    if total_repositioning:
        helped_or_partly = repositioning_counts.get("YES", 0) + repositioning_counts.get("PARTLY", 0)
        derived["repositioning_helped_or_partly_ratio"] = round(helped_or_partly / total_repositioning, 3)

    skin_counts = field_value_counter.get("skin_pressure_concern_noted", Counter())
    if skin_counts:
        derived["skin_pressure_concerns_reported"] = {
            k: v for k, v in skin_counts.items() if k != "NO_CONCERN"
        }

    med_timing_counts = field_value_counter.get("medication_timing_relation", Counter())
    if med_timing_counts:
        derived["medication_timing_context_completion_count"] = sum(
            v for k, v in med_timing_counts.items() if k != "NOT_SURE"
        )

    return derived

NEXT_CYCLE_DERIVED_FEATURES = {
    pid: derive_features_from_responses(pid, NEXT_CYCLE_RESPONSE_EVENTS)
    for pid in PATIENT_PROFILES
}

with open(OUTPUT_DIR / "uc4_next_cycle_derived_features_from_caregiver_responses.json", "w") as f:
    json.dump(NEXT_CYCLE_DERIVED_FEATURES, f, indent=2)

NEXT_CYCLE_DERIVED_FEATURES

{'Mike_DEMO_001': {'patient_id': 'Mike_DEMO_001',
  'response_event_count': 5,
  'template_response_counts': {'SKIN_PRESSURE_AFTER_SEATED_PERIOD': 1,
   'MEDICATION_WINDOW_FATIGUE_TRACKING': 1,
   'BOWEL_ROUTINE_DISCOMFORT_CONTEXT': 1,
   'TRANSFER_DISCOMFORT_TRACKING': 1,
   'MISSED_DELAYED_MEDICATION_CONTEXT': 1},
  'field_value_counts': {'long_seated_period': {'YES': 1},
   'seated_duration_estimate': {'NOT_SURE': 1},
   'skin_pressure_check_completed': {'YES': 1},
   'skin_pressure_concern_noted': {'NO_CONCERN': 1},
   'repositioning_occurred': {'YES': 1},
   'fatigue_start_time_estimate': {'EVENING': 1},
   'medication_timing_relation': {'NOT_SURE': 1},
   'dose_missed_or_delayed': {'NOT_SURE': 1},
   'differs_from_baseline': {'YES_DIFFERENT': 2},
   'nearby_concerns': {'POOR_SLEEP_OR_RESTLESSNESS': 1,
    'UNUSUAL_RESPONSIVENESS': 1},
   'bowel_bladder_routine_change': {'NOT_SURE': 1},
   'hydration_or_intake_context': {'NOT_SURE': 1},
   'discomfort_timing': {'BEFORE_BATHROOM_RO

In [50]:
def render_provider_summary_for_patient(patient_id, cards):
    profile = PATIENT_PROFILES[patient_id]
    lines = []
    lines.append(f"UC4 Structured Micro-Priority Summary")
    lines.append(f"Patient: {profile['display_name']} ({patient_id})")
    lines.append(f"Profile context: {profile['profile_label']}")
    lines.append(f"Generated at: {iso(NOW)}")
    lines.append("")
    lines.append("Safety boundary:")
    lines.append("- UC4 provides structured caregiver observation support only.")
    lines.append("- It does not diagnose, infer medication causality, detect seizures/wounds, or recommend treatment/medication changes.")
    lines.append("")

    for card in cards:
        lines.append(f"Priority {card['rank']}: {card['title']}")
        lines.append(f"Template: {card['template_id']}")
        lines.append(f"Domain: {card['domain']}")
        lines.append(f"Score: {card['scores']['final_score']}")
        lines.append("Why selected:")
        for w in card["why_this_is_on_the_list"]:
            lines.append(f"  - {w}")
        lines.append("Rules fired:")
        for r in card["evidence"]["rules_fired"]:
            lines.append(f"  - {r}")
        lines.append("Structured fields requested:")
        for f in card["what_to_log_next_schema"]:
            lines.append(f"  - {f['field_id']} ({f['input_type']})")
        lines.append("")

    return "\n".join(lines)

PROVIDER_SUMMARIES = {
    pid: render_provider_summary_for_patient(pid, cards)
    for pid, cards in PRIORITY_CARDS.items()
}

for pid, summary in PROVIDER_SUMMARIES.items():
    with open(OUTPUT_DIR / f"uc4_provider_summary_{pid}.txt", "w") as f:
        f.write(summary)

print(PROVIDER_SUMMARIES["Mike_DEMO_001"])

UC4 Structured Micro-Priority Summary
Patient: Mike (Mike_DEMO_001)
Profile context: Cerebral palsy, caregiver-supported home routines
Generated at: 2026-07-16T12:00:00Z

Safety boundary:
- UC4 provides structured caregiver observation support only.
- It does not diagnose, infer medication causality, detect seizures/wounds, or recommend treatment/medication changes.

Priority 1: Add a quick skin/pressure check after long seated periods
Template: SKIN_PRESSURE_AFTER_SEATED_PERIOD
Domain: Skin and positioning support
Score: 0.805
Why selected:
  - Patient profile indicates high mobility support.
  - Low-movement hours/day: 9.5. Baseline: 7.0.
  - No skin/pressure concern logs were recorded in the past 7 days.
Rules fired:
  - R_HIGH_MOBILITY_SUPPORT
  - R_LOW_MOVEMENT_INCREASE
  - R_SKIN_LOG_MISSING
Structured fields requested:
  - long_seated_period (single_select)
  - seated_duration_estimate (single_select)
  - skin_pressure_check_completed (single_select)
  - skin_pressure_concern_no

In [52]:
APP_PAYLOADS = {}

for pid, cards in PRIORITY_CARDS.items():
    APP_PAYLOADS[pid] = {
        "patient_id": pid,
        "patient_display_name": PATIENT_PROFILES[pid]["display_name"],
        "generated_at": iso(NOW),
        "engine_version": ENGINE_VERSION,
        "schema_version": SCHEMA_VERSION,
        "template_registry_version": TEMPLATE_REGISTRY_VERSION,
        "mode": "STRUCTURED_RULE_TEMPLATE_ENGINE",
        "slm_used": False,
        "free_text_used_for_scoring": False,
        "priorities": cards,
        "provider_summary_path": f"uc4_provider_summary_{pid}.txt",
        "safety_global": {
            "diagnosis_made": False,
            "medication_change_recommended": False,
            "treatment_change_recommended": False,
            "medication_causality_claimed": False,
            "seizure_detected": False,
            "wound_detected": False,
            "tone_or_spasticity_measured": False
        }
    }

with open(OUTPUT_DIR / "uc4_app_payloads_by_patient.json", "w") as f:
    json.dump(APP_PAYLOADS, f, indent=2)

# Individual payloads
for pid, payload in APP_PAYLOADS.items():
    with open(OUTPUT_DIR / f"uc4_app_payload_{pid}.json", "w") as f:
        json.dump(payload, f, indent=2)

APP_PAYLOADS.keys()

dict_keys(['Mike_DEMO_001', 'JAMES_DEMO_001', 'SOFIA_DEMO_001', 'ELENA_DEMO_001'])

In [54]:
AUDIT_RECORDS = []

for pid, cards in PRIORITY_CARDS.items():
    for card in cards:
        AUDIT_RECORDS.append({
            "audit_id": f"audit_{card['priority_id']}",
            "patient_id": pid,
            "priority_id": card["priority_id"],
            "template_id": card["template_id"],
            "generated_at": iso(NOW),
            "engine_version": ENGINE_VERSION,
            "schema_version": SCHEMA_VERSION,
            "template_registry_version": TEMPLATE_REGISTRY_VERSION,
            "rules_fired": card["evidence"]["rules_fired"],
            "scores": card["scores"],
            "safety_flags": card["safety_flags"],
            "free_text_used_for_scoring": False,
            "slm_used": False,
            "what_to_log_next_schema_field_ids": [
                f["field_id"] for f in card["what_to_log_next_schema"]
            ]
        })

with open(OUTPUT_DIR / "uc4_audit_records.json", "w") as f:
    json.dump(AUDIT_RECORDS, f, indent=2)

pd.DataFrame(AUDIT_RECORDS).head()

,audit_id,patient_id,priority_id,template_id,generated_at,engine_version,schema_version,template_registry_version,rules_fired,scores,safety_flags,free_text_used_for_scoring,slm_used,what_to_log_next_schema_field_ids
0,audit_uc4_priority_Mike_DEMO_001_01_SKIN_PRESS...,Mike_DEMO_001,uc4_priority_Mike_DEMO_001_01_SKIN_PRESSURE_AF...,SKIN_PRESSURE_AFTER_SEATED_PERIOD,2026-07-16T12:00:00Z,uc4_structured_micropriority_engine_v0.1.0,uc4_schema_v0.1.0,uc4_template_registry_v0.1.0,"[R_HIGH_MOBILITY_SUPPORT, R_LOW_MOVEMENT_INCRE...","{'specificity_score': 0.9, 'recurrence_score':...","{'diagnosis_made': False, 'medication_change_r...",False,False,"[long_seated_period, seated_duration_estimate,..."
1,audit_uc4_priority_Mike_DEMO_001_02_MEDICATION...,Mike_DEMO_001,uc4_priority_Mike_DEMO_001_02_MEDICATION_WINDO...,MEDICATION_WINDOW_FATIGUE_TRACKING,2026-07-16T12:00:00Z,uc4_structured_micropriority_engine_v0.1.0,uc4_schema_v0.1.0,uc4_template_registry_v0.1.0,"[R_FATIGUE_RECURRENCE, R_MED_WATCH_AREA_MATCH,...","{'specificity_score': 0.9, 'recurrence_score':...","{'diagnosis_made': False, 'medication_change_r...",False,False,"[fatigue_start_time_estimate, medication_timin..."
2,audit_uc4_priority_Mike_DEMO_001_03_BOWEL_ROUT...,Mike_DEMO_001,uc4_priority_Mike_DEMO_001_03_BOWEL_ROUTINE_DI...,BOWEL_ROUTINE_DISCOMFORT_CONTEXT,2026-07-16T12:00:00Z,uc4_structured_micropriority_engine_v0.1.0,uc4_schema_v0.1.0,uc4_template_registry_v0.1.0,"[R_BOWEL_CHANGE_LOGGED, R_DISCOMFORT_OVERLAP, ...","{'specificity_score': 0.9, 'recurrence_score':...","{'diagnosis_made': False, 'medication_change_r...",False,False,"[bowel_bladder_routine_change, hydration_or_in..."
3,audit_uc4_priority_Mike_DEMO_001_04_TRANSFER_D...,Mike_DEMO_001,uc4_priority_Mike_DEMO_001_04_TRANSFER_DISCOMF...,TRANSFER_DISCOMFORT_TRACKING,2026-07-16T12:00:00Z,uc4_structured_micropriority_engine_v0.1.0,uc4_schema_v0.1.0,uc4_template_registry_v0.1.0,"[R_REPEATED_DISCOMFORT, R_TRANSFER_CONTEXT_CLU...","{'specificity_score': 0.9, 'recurrence_score':...","{'diagnosis_made': False, 'medication_change_r...",False,False,"[transfer_timing, transfer_phase, position_bef..."
4,audit_uc4_priority_Mike_DEMO_001_05_MISSED_DEL...,Mike_DEMO_001,uc4_priority_Mike_DEMO_001_05_MISSED_DELAYED_M...,MISSED_DELAYED_MEDICATION_CONTEXT,2026-07-16T12:00:00Z,uc4_structured_micropriority_engine_v0.1.0,uc4_schema_v0.1.0,uc4_template_registry_v0.1.0,"[R_MISSED_DELAYED_MEDICATION_LOGGED, R_MEDICAT...","{'specificity_score': 0.65, 'recurrence_score'...","{'diagnosis_made': False, 'medication_change_r...",False,False,"[medication_identifier, dose_status, approxima..."


In [56]:
def validate_template_registry(registry):
    required_template_keys = [
        "template_id",
        "title_template",
        "priority_type",
        "domain",
        "trigger_rules",
        "what_to_watch",
        "what_to_log_next_display",
        "what_to_log_next_schema",
        "when_to_share_or_escalate",
        "safety_boundary",
        "safety_flags"
    ]

    required_field_keys = ["field_id", "label", "input_type", "required"]

    for tid, tmpl in registry.items():
        for key in required_template_keys:
            assert key in tmpl, f"{tid} missing template key: {key}"

        assert tmpl["template_id"] == tid, f"{tid} template_id mismatch"
        assert len(tmpl["what_to_log_next_schema"]) > 0, f"{tid} has no structured what_to_log_next_schema"

        for f in tmpl["what_to_log_next_schema"]:
            for key in required_field_keys:
                assert key in f, f"{tid} field missing {key}: {f}"

            assert f["input_type"] in INPUT_TYPES, f"{tid} invalid input_type: {f['input_type']}"

            if f["input_type"] in ["single_select", "multi_select"]:
                assert "options" in f and len(f["options"]) > 0, f"{tid} select field missing options: {f['field_id']}"

        safety = tmpl["safety_flags"]
        assert safety["diagnosis_made"] is False
        assert safety["medication_change_recommended"] is False
        assert safety["treatment_change_recommended"] is False
        assert safety["medication_causality_claimed"] is False
        assert safety["seizure_detected"] is False
        assert safety["wound_detected"] is False
        assert safety["tone_or_spasticity_measured"] is False

def validate_priority_outputs(priority_cards):
    for pid, cards in priority_cards.items():
        for card in cards:
            assert card["template_id"] in TEMPLATE_REGISTRY, f"Unknown template_id: {card['template_id']}"
            assert "what_to_log_next_schema" in card, f"Missing schema in card {card['priority_id']}"
            assert len(card["what_to_log_next_schema"]) > 0, f"Empty schema in card {card['priority_id']}"
            assert "what_to_log_next_display" in card
            assert card["audit"]["free_text_used_for_scoring"] is False
            assert card["audit"]["slm_used"] is False

            for flag, value in card["safety_flags"].items():
                assert value is False, f"Safety flag violation in {card['priority_id']}: {flag}={value}"

def validate_caregiver_response_examples(response_examples):
    for pid, responses in response_examples.items():
        for r in responses:
            assert r["template_id"] in TEMPLATE_REGISTRY
            schema_fields = {
                f["field_id"]
                for f in TEMPLATE_REGISTRY[r["template_id"]]["what_to_log_next_schema"]
            }
            response_fields = set(r["responses"].keys())
            assert response_fields.issubset(schema_fields), f"Response has fields not in schema: {response_fields - schema_fields}"
            assert r["free_text_used_for_scoring"] is False
            assert r["audit"]["slm_used"] is False

validate_template_registry(TEMPLATE_REGISTRY)
validate_priority_outputs(PRIORITY_CARDS)
validate_caregiver_response_examples(CAREGIVER_RESPONSE_EXAMPLES)

print("All UC4 structural validation checks passed.")

All UC4 structural validation checks passed.


In [58]:
manifest = {
    "notebook_name": "UC4_Structured_MicroPriority_Engine_Generalizable.ipynb",
    "created_at": iso(NOW),
    "engine_version": ENGINE_VERSION,
    "schema_version": SCHEMA_VERSION,
    "template_registry_version": TEMPLATE_REGISTRY_VERSION,
    "patients_supported_in_demo": list(PATIENT_PROFILES.keys()),
    "output_files": [
        "uc4_controlled_vocabularies.json",
        "uc4_option_sets.json",
        "uc4_template_registry.json",
        "synthetic_patient_profiles.json",
        "synthetic_medication_profiles.json",
        "synthetic_medication_profiles.csv",
        "observation_to_medication_watch_area_map.json",
        "synthetic_shared_care_events.csv",
        "synthetic_wearable_weekly_summaries.json",
        "previous_uc4_priorities.json",
        "uc4_patient_aggregates.json",
        "uc4_micro_priority_candidates.csv",
        "uc4_top_structured_priorities.csv",
        "uc4_structured_priority_cards_by_patient.json",
        "uc4_caregiver_response_examples.json",
        "uc4_next_cycle_response_events.json",
        "uc4_next_cycle_derived_features_from_caregiver_responses.json",
        "uc4_app_payloads_by_patient.json",
        "uc4_audit_records.json",
        "uc4_provider_summary_Mike_DEMO_001.txt",
        "uc4_provider_summary_JAMES_DEMO_001.txt",
        "uc4_provider_summary_SOFIA_DEMO_001.txt",
        "uc4_provider_summary_ELENA_DEMO_001.txt"
    ],
    "safety_boundaries": [
        "No diagnosis",
        "No medication causality inference",
        "No medication change recommendation",
        "No treatment recommendation",
        "No seizure detection",
        "No wound detection",
        "No tone/spasticity measurement",
        "No free text used for scoring",
        "No SLM used for scoring"
    ],
    "non_negotiable_schema_rule": "Every caregiver-facing what_to_log_next item is backed by a structured input schema."
}

with open(OUTPUT_DIR / "output_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

manifest

{'notebook_name': 'UC4_Structured_MicroPriority_Engine_Generalizable.ipynb',
 'created_at': '2026-07-16T12:00:00Z',
 'engine_version': 'uc4_structured_micropriority_engine_v0.1.0',
 'schema_version': 'uc4_schema_v0.1.0',
 'template_registry_version': 'uc4_template_registry_v0.1.0',
 'patients_supported_in_demo': ['Mike_DEMO_001',
  'JAMES_DEMO_001',
  'SOFIA_DEMO_001',
  'ELENA_DEMO_001'],
 'output_files': ['uc4_controlled_vocabularies.json',
  'uc4_option_sets.json',
  'uc4_template_registry.json',
  'synthetic_patient_profiles.json',
  'synthetic_medication_profiles.json',
  'synthetic_medication_profiles.csv',
  'observation_to_medication_watch_area_map.json',
  'synthetic_shared_care_events.csv',
  'synthetic_wearable_weekly_summaries.json',
  'previous_uc4_priorities.json',
  'uc4_patient_aggregates.json',
  'uc4_micro_priority_candidates.csv',
  'uc4_top_structured_priorities.csv',
  'uc4_structured_priority_cards_by_patient.json',
  'uc4_caregiver_response_examples.json',
  'uc4